# Nyay AI — Final RAG Evaluation & UI Integration Notebook

Maharashtra-focused legal RAG assistant.

Architecture:
1. Ingestion Agent
2. Retriever Agent
3. Answer Generation Agent (Gemini)
4. Citation & Confidence Agent
5. Orchestrator
6. User-upload workspace / unified official-user retrieval

Validated official evaluation artifacts are stored under the evaluation directory.


In [1]:
# Cell 1 — Install dependencies (Colab)
!pip -q install pymupdf langchain langchain-core langchain-huggingface langchain-chroma chromadb sentence-transformers transformers pandas numpy scikit-learn matplotlib openai

print("✅ Dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [2]:
# Cell 2 — Drive, paths, and configuration
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, re, json, time, math
import numpy as np
import pandas as pd

DATASET_ROOT = Path("/content/drive/MyDrive/datset 2")
PROJECT_ROOT = Path("/content/drive/MyDrive/Nyay_AI")
DRIVE_CHROMA_PATH = PROJECT_ROOT / "chroma_final"
USER_UPLOAD_CHROMA_PATH = PROJECT_ROOT / "user_upload_chroma"
EVALUATION_PATH = PROJECT_ROOT / "evaluation"
ARTIFACT_PATH = PROJECT_ROOT / "artifacts"

for p in [PROJECT_ROOT, USER_UPLOAD_CHROMA_PATH, EVALUATION_PATH, ARTIFACT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_PDFS = 68
EXPECTED_CHUNKS = 5259
COLLECTION_NAME = "nyay_ai_final"

TOKENIZER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_MODEL = TOKENIZER_NAME
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 5
CONFIDENCE_THRESHOLD = 0.70

LLM_PROVIDER = "gemini"
LLM_MODEL = "gemini-3.6-flash"

print("✅ Configuration loaded")


Mounted at /content/drive
✅ Configuration loaded


In [3]:
# Cell 3 — Validate the official 68-PDF corpus
pdfs = sorted(set(
    [p.resolve() for p in DATASET_ROOT.rglob("*.pdf")] +
    [p.resolve() for p in DATASET_ROOT.rglob("*.PDF")]
))

print("PDFs found:", len(pdfs))
assert len(pdfs) == EXPECTED_PDFS, f"Expected {EXPECTED_PDFS}, found {len(pdfs)}"
print("✅ Official corpus contains exactly 68 PDFs")


PDFs found: 68
✅ Official corpus contains exactly 68 PDFs


In [4]:
# Cell 4 — Official corpus inventory
inventory = []
for idx, pdf_path in enumerate(pdfs, start=1):
    inventory.append({
        "document_id": f"DOC_{idx:03d}",
        "filename": pdf_path.name,
        "relative_path": str(pdf_path.relative_to(DATASET_ROOT))
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df.head(10))
print("Total documents:", len(inventory_df))


,document_id,filename,relative_path
0,DOC_001,Checklists for CriminalCivil Proposals.pdf,Circulars and Government Resolutions issued by...
1,DOC_002,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...
2,DOC_003,Procedure in case reference made to the Rememb...,Circulars and Government Resolutions issued by...
3,DOC_004,Rules of procedure for dealing of Legislation ...,Circulars and Government Resolutions issued by...
4,DOC_005,The Maharashtra Agricultural Produce Marketing...,Maharashtra Acts/Business/The Maharashtra Agri...
5,DOC_006,"The Maharashtra Co-operative Societies Act, 19...",Maharashtra Acts/Business/The Maharashtra Co-o...
6,DOC_007,"The Maharashtra State Tax on Professions, Trad...",Maharashtra Acts/Business/The Maharashtra Stat...
7,DOC_008,"The Maharashtra Value Added Tax Act, 2002..pdf",Maharashtra Acts/Business/The Maharashtra Valu...
8,DOC_009,"Maharashtra Employees of Private Schools Act, ...",Maharashtra Acts/Labour/Maharashtra Employees ...
9,DOC_010,Maharashtra Industrial Relations Act.pdf,Maharashtra Acts/Labour/Maharashtra Industrial...


Total documents: 68


In [5]:
# Cell 5 — Tokenizer and chunk settings
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print("Tokenizer:", TOKENIZER_NAME)
print("Chunk size:", CHUNK_SIZE, "tokens")
print("Overlap:", CHUNK_OVERLAP, "tokens")
print("MiniLM max position embeddings:", getattr(tokenizer, "model_max_length", "unknown"))


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer: sentence-transformers/all-MiniLM-L6-v2
Chunk size: 500 tokens
Overlap: 50 tokens
MiniLM max position embeddings: 512


In [6]:
# Cell 6 — Text cleaning + sentence-boundary-aware 500/50 chunking
def clean_pdf_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Conservative page-number cleanup only; do not delete legal text heuristically.
    text = re.sub(r"(?m)^\s*(?:page\s+)?\d+\s*$", "", text, flags=re.IGNORECASE)
    return text.strip()

def sentence_token_chunks(text, tokenizer, chunk_size=500, overlap=50):
    """Token budget is 500 with 50-token overlap; sentence boundaries are preferred."""
    text = clean_pdf_text(text)
    if not text:
        return []

    # Preserve sentence-ish units, then pack them into <= chunk_size tokens.
    units = [u.strip() for u in re.split(r"(?<=[.!?])\s+|\n+", text) if u.strip()]
    chunks, current = [], []
    current_tokens = 0

    def tok_len(s):
        return len(tokenizer.encode(s, add_special_tokens=False))

    for unit in units:
        n = tok_len(unit)
        if n > chunk_size:
            ids = tokenizer.encode(unit, add_special_tokens=False)
            start = 0
            while start < len(ids):
                piece_ids = ids[start:start + chunk_size]
                piece = tokenizer.decode(piece_ids, skip_special_tokens=True).strip()
                if piece:
                    chunks.append(piece)
                start += chunk_size - overlap
            current, current_tokens = [], 0
            continue

        if current and current_tokens + n > chunk_size:
            chunk_text = " ".join(current).strip()
            chunks.append(chunk_text)

            # Build overlap from the end of the previous chunk.
            overlap_ids = tokenizer.encode(chunk_text, add_special_tokens=False)[-overlap:]
            overlap_text = tokenizer.decode(overlap_ids, skip_special_tokens=True).strip()
            current = [overlap_text] if overlap_text else []
            current_tokens = tok_len(overlap_text) if overlap_text else 0

        current.append(unit)
        current_tokens += n

    if current:
        chunks.append(" ".join(current).strip())

    return [c for c in chunks if c]

print("✅ Chunking function ready")


✅ Chunking function ready


In [7]:
# Cell 7 — Ingestion Agent
import pymupdf
from langchain_core.documents import Document

class IngestionAgent:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def extract_and_chunk(self, pdf_path, metadata=None):
        pdf_path = Path(pdf_path)
        metadata = dict(metadata or {})
        documents = []

        pdf = pymupdf.open(str(pdf_path))
        try:
            for page_index in range(len(pdf)):
                page = pdf[page_index]
                raw = page.get_text("text")
                text = clean_pdf_text(raw)
                if not text:
                    continue

                page_chunks = sentence_token_chunks(
                    text, self.tokenizer, CHUNK_SIZE, CHUNK_OVERLAP
                )

                for chunk_index, chunk_text in enumerate(page_chunks):
                    md = dict(metadata)
                    md.update({
                        "filename": pdf_path.name,
                        "page": page_index + 1,
                        "chunk_index": chunk_index,
                        "source": pdf_path.name,
                    })
                    documents.append(Document(page_content=chunk_text, metadata=md))
        finally:
            pdf.close()

        return documents

ingestion_agent = IngestionAgent(tokenizer)
print("✅ Ingestion Agent ready")


✅ Ingestion Agent ready


In [8]:
# Cell 8 — Embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Explicit dimension check
probe = embeddings.embed_query("Maharashtra legal question")
print("Embedding dimension:", len(probe))
assert len(probe) == 384
print("✅ 384-dimensional normalized embeddings confirmed")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
✅ 384-dimensional normalized embeddings confirmed


In [9]:
# Cell 9 — Load existing official ChromaDB
from langchain_chroma import Chroma

assert DRIVE_CHROMA_PATH.exists(), f"Official ChromaDB not found: {DRIVE_CHROMA_PATH}"

official_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(DRIVE_CHROMA_PATH),
    embedding_function=embeddings,
)

official_count = official_store._collection.count()
print("Official ChromaDB records:", official_count)
assert official_count > 0
if official_count != EXPECTED_CHUNKS:
    print(f"⚠️ Expected previous validated count {EXPECTED_CHUNKS}, found {official_count}. Do not rebuild automatically.")
else:
    print("✅ Existing 5,259-record official index loaded")


Official ChromaDB records: 5259
✅ Existing 5,259-record official index loaded


In [10]:
# Cell 10 — Official index metadata/token safety sample
sample_docs = official_store.similarity_search("Maharashtra law", k=min(20, official_count))
sample_lengths = [
    len(tokenizer.encode(d.page_content, add_special_tokens=False))
    for d in sample_docs
]
print("Sample max tokens:", max(sample_lengths) if sample_lengths else 0)
print("Sample average tokens:", round(float(np.mean(sample_lengths)), 2) if sample_lengths else 0)

if sample_lengths and max(sample_lengths) > 512:
    print("⚠️ A sampled stored chunk exceeds MiniLM's 512-token limit. The existing index should be rebuilt before claiming strict token safety.")
else:
    print("✅ Sampled stored chunks fit the MiniLM limit")


Sample max tokens: 500
Sample average tokens: 327.75
✅ Sampled stored chunks fit the MiniLM limit


In [46]:
# Cell 11 — Robust cosine top-5 Retriever Agent
class RetrieverAgent:
    def __init__(self, vectorstore, embeddings, k=5, name="official"):
        self.vectorstore = vectorstore
        self.embeddings = embeddings
        self.k = k
        self.name = name

    def retrieve(self, question):
        # Retrieve a candidate pool, then explicitly rank by cosine similarity.
        candidate_k = max(self.k, 5)
        docs = self.vectorstore.similarity_search(question, k=candidate_k)
        if not docs:
            return []

        q = np.asarray(self.embeddings.embed_query(question), dtype=float)
        q_norm = np.linalg.norm(q)

        scored = []
        for doc in docs:
            v = np.asarray(self.embeddings.embed_query(doc.page_content), dtype=float)
            denom = q_norm * np.linalg.norm(v)
            sim = float(np.dot(q, v) / denom) if denom else 0.0
            doc.metadata = dict(doc.metadata)
            doc.metadata["similarity"] = sim
            doc.metadata["retrieval_source"] = self.name
            scored.append(doc)

        return sorted(scored, key=lambda d: d.metadata["similarity"], reverse=True)[:self.k]

official_retriever = RetrieverAgent(
    official_store, embeddings, TOP_K, name="official_corpus"
)
print("✅ Official Retriever Agent ready: cosine top-5")


✅ Official Retriever Agent ready: cosine top-5


In [12]:
# Cell 12 — Citation + confidence helpers
CITATION_PATTERN = re.compile(r"\[S(\d+)\]")

def source_cards(retrieved_docs):
    cards = []
    for i, doc in enumerate(retrieved_docs, 1):
        md = doc.metadata
        cards.append({
            "source_id": f"S{i}",
            "filename": md.get("filename", "Unknown"),
            "law_name": md.get("law_name", md.get("filename", "Unknown")),
            "page": md.get("page", "Unknown"),
            "document_id": md.get("document_id", "Unknown"),
            "retrieval_source": md.get("retrieval_source", "unknown"),
            "similarity": float(md.get("similarity", 0.0)),
            "excerpt": doc.page_content,
        })
    return cards

def verify_generated_citations(answer, retrieved_docs):
    cited = sorted(set(int(x) for x in CITATION_PATTERN.findall(answer or "")))
    valid = set(range(1, len(retrieved_docs) + 1))
    unsupported = sorted(set(cited) - valid)
    return {
        "cited_source_ids": [f"S{x}" for x in cited],
        "verified_source_ids": [f"S{x}" for x in cited if x in valid],
        "unsupported_source_ids": [f"S{x}" for x in unsupported],
        "citation_format_accuracy": (
            len([x for x in cited if x in valid]) / len(cited)
            if cited else 0.0
        ),
    }

class CitationConfidenceAgent:
    CONFIDENCE_THRESHOLD = 0.70

    def process(self, answer, retrieved_docs):
        max_similarity = max(
            [float(d.metadata.get("similarity", 0.0)) for d in retrieved_docs],
            default=0.0
        )
        insufficient = (
            not retrieved_docs or
            "i could not find this information in the loaded documents" in answer.lower()
        )
        confidence = (
            "Corpus-Grounded"
            if (max_similarity >= self.CONFIDENCE_THRESHOLD and not insufficient)
            else "Insufficient Evidence"
        )
        return {
            "answer": answer,
            "confidence": confidence,
            "max_similarity": max_similarity,
            "citations": source_cards(retrieved_docs),
        }

citation_agent = CitationConfidenceAgent()
print("✅ Citation & Confidence Agent ready")


✅ Citation & Confidence Agent ready


In [38]:
# Cell 13 — Gemini client

from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets."
    )

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODEL = "gemini-3.6-flash"

print("Gemini client initialized")
print("Model:", GEMINI_MODEL)

Gemini client initialized
Model: gemini-3.6-flash


In [39]:
# Cell 14 — Gemini connection test

test_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Reply with exactly: OK"
)

print("Gemini response:", test_response.text)

Gemini response: OK


In [40]:
# Cell 15 — Gemini Answer Generation Agent with retry

import re
import time

class AnswerGenerationAgent:
    def __init__(self, client, model, max_retries=3):
        self.client = client
        self.model = model
        self.max_retries = max_retries

    def generate(self, question, retrieved_chunks):

        if not retrieved_chunks:
            return {
                "answer": "Insufficient Evidence",
                "confidence": 0.0,
                "citations": [],
                "input_tokens": 0,
                "output_tokens": 0,
                "total_tokens": 0
            }

        context_parts = []

        for i, doc in enumerate(retrieved_chunks, start=1):
            metadata = doc.metadata or {}

            law_name = metadata.get(
                "law_name",
                metadata.get("source", "Unknown Source")
            )

            page = metadata.get(
                "page",
                metadata.get("page_number", "Unknown")
            )

            context_parts.append(
                f"[S{i}] {law_name}, page {page}\n"
                f"{doc.page_content}"
            )

        context = "\n\n".join(context_parts)

        system_instruction = """
You are Nyay AI, a Maharashtra-focused legal RAG assistant.

Answer ONLY using the supplied retrieved legal sources.

Rules:
1. Do not invent facts, sections, cases, dates, or citations.
2. If the retrieved sources do not contain enough evidence, say:
   "Insufficient Evidence"
3. Give a concise and legally clear answer.
4. Cite supporting sources using [S1], [S2], etc.
5. Do not cite a source unless it supports the statement.
6. Prefer the retrieved Maharashtra legal documents over general knowledge.
"""

        prompt = f"""
{system_instruction}

RETRIEVED LEGAL SOURCES:
{context}

QUESTION:
{question}

ANSWER:
"""

        # Retry temporary Gemini 503 errors
        response = None

        for attempt in range(self.max_retries):
            try:
                response = self.client.models.generate_content(
                    model=self.model,
                    contents=prompt
                )
                break

            except Exception as e:
                error_text = str(e)

                if "503" in error_text or "UNAVAILABLE" in error_text:
                    if attempt < self.max_retries - 1:
                        wait_time = 5 * (attempt + 1)

                        print(
                            f"Gemini temporarily unavailable. "
                            f"Retrying in {wait_time}s..."
                        )

                        time.sleep(wait_time)
                    else:
                        raise

                else:
                    raise

        answer = response.text.strip()

        # Gemini token usage
        usage = getattr(response, "usage_metadata", None)

        input_tokens = 0
        output_tokens = 0
        total_tokens = 0

        if usage:
            input_tokens = getattr(
                usage,
                "prompt_token_count",
                0
            ) or 0

            output_tokens = getattr(
                usage,
                "candidates_token_count",
                0
            ) or 0

            total_tokens = getattr(
                usage,
                "total_token_count",
                0
            ) or (input_tokens + output_tokens)

        # Extract [S1], [S2], etc.
        citation_numbers = sorted(
            set(re.findall(r"\[S(\d+)\]", answer))
        )

        citations = [
            f"S{x}" for x in citation_numbers
        ]

        if answer == "Insufficient Evidence":
            confidence = 0.0
        elif citations:
            confidence = 1.0
        else:
            confidence = 0.0

        return {
            "answer": answer,
            "confidence": confidence,
            "citations": citations,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens
        }


answer_agent = AnswerGenerationAgent(
    client=gemini_client,
    model=GEMINI_MODEL,
    max_retries=3
)

print("Gemini AnswerGenerationAgent ready")

Gemini AnswerGenerationAgent ready


In [47]:
# Cell 16 — Official Orchestrator
class NyayAIOrchestrator:
    def __init__(self, retriever, answer_generator, citation_agent):
        self.retriever = retriever
        self.answer_generator = answer_generator
        self.citation_agent = citation_agent

    def run(self, question):
        start = time.perf_counter()
        docs = self.retriever.retrieve(question)
        generation = self.answer_generator.generate(question, docs)
        citation_result = self.citation_agent.process(generation["answer"], docs)
        citation_check = verify_generated_citations(generation["answer"], docs)
        latency = time.perf_counter() - start

        return {
            "question": question,
            "answer": generation["answer"],
            "confidence": citation_result["confidence"],
            "max_similarity": citation_result["max_similarity"],
            "citations": citation_result["citations"],
            "citation_check": citation_check,
            "retrieved_documents": docs,
            "input_tokens": generation["input_tokens"],
            "output_tokens": generation["output_tokens"],
            "latency_seconds": round(latency, 3),
        }

official_orchestrator = NyayAIOrchestrator(
    official_retriever, answer_agent, citation_agent
)
print("✅ Official Orchestrator ready")


✅ Official Orchestrator ready


In [42]:
# Cell 17 — Official end-to-end sanity test

q = "What is the purpose of the Maharashtra Rent Control Act, 1999?"
r = official_orchestrator.run(q)

print("QUESTION:", q)
print("\nANSWER:\n", r["answer"])
print("\nCONFIDENCE:", r["confidence"])
print("MAX SIMILARITY:", round(r["max_similarity"], 4))
print("LATENCY:", r["latency_seconds"], "sec")
print("CITATIONS:", r["citation_check"])

QUESTION: What is the purpose of the Maharashtra Rent Control Act, 1999?

ANSWER:
 The purpose of the Maharashtra Rent Control Act, 1999 is to unify, consolidate, and amend the laws prevailing in different parts of the State of Maharashtra relating to:

* The control of rent and repairs of certain premises [S2].
* Eviction from premises [S2].
* Encouraging the construction of new houses by assuring a fair return on investment to landlords [S2].
* Matters connected with these purposes [S2].

CONFIDENCE: Corpus-Grounded
MAX SIMILARITY: 0.8963
LATENCY: 57.362 sec
CITATIONS: {'cited_source_ids': ['S2'], 'verified_source_ids': ['S2'], 'unsupported_source_ids': [], 'citation_format_accuracy': 1.0}


In [26]:
# Cell 18 — User Upload workspace + UserUploadAgent
from langchain_chroma import Chroma

def get_user_store(workspace_id):
    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "_", str(workspace_id))
    path = USER_UPLOAD_CHROMA_PATH / safe_id
    path.mkdir(parents=True, exist_ok=True)
    return Chroma(
        collection_name=f"user_{safe_id}",
        persist_directory=str(path),
        embedding_function=embeddings,
    )

class UserUploadAgent:
    def __init__(self, workspace_id):
        self.workspace_id = str(workspace_id)
        self.store = get_user_store(self.workspace_id)

    def add_pdf(self, pdf_path):
        docs = ingestion_agent.extract_and_chunk(
            pdf_path,
            metadata={
                "category": "user_upload",
                "doc_type": "User PDF",
                "jurisdiction": "User document",
                "law_name": Path(pdf_path).name,
            },
        )
        if docs:
            self.store.add_documents(docs)
        return {
            "workspace_id": self.workspace_id,
            "filename": Path(pdf_path).name,
            "chunks_added": len(docs),
        }

    def retrieve(self, question, k=5):
        # Same explicit cosine ranking used by the official retriever.
        return RetrieverAgent(
            self.store, embeddings, k=k, name="user_upload"
        ).retrieve(question)

print("✅ User Upload Agent ready")


✅ User Upload Agent ready


In [27]:
# Cell 19 — Unified Retriever + Orchestrator for official/user/both modes
class UnifiedRetrieverAgent:
    def __init__(self, official_retriever, user_agent=None, k=5):
        self.official = official_retriever
        self.user_agent = user_agent
        self.k = k

    def retrieve(self, question, mode="both"):
        mode = mode.lower()
        docs = []

        if mode in ("official", "both"):
            docs.extend(self.official.retrieve(question))

        if mode in ("user", "both"):
            if self.user_agent is None:
                raise ValueError("UserUploadAgent is required for user/both mode.")
            docs.extend(self.user_agent.retrieve(question, k=self.k))

        if not docs:
            return []

        # Re-rank the union by cosine similarity and keep the global top-5.
        docs = sorted(
            docs,
            key=lambda d: float(d.metadata.get("similarity", 0.0)),
            reverse=True,
        )

        # Avoid duplicate chunk IDs where possible.
        unique = []
        seen = set()
        for d in docs:
            md = d.metadata
            key = (
                md.get("retrieval_source"),
                md.get("filename"),
                md.get("page"),
                md.get("chunk_index"),
                d.page_content[:80],
            )
            if key not in seen:
                seen.add(key)
                unique.append(d)

        return unique[:self.k]

class UnifiedNyayAIOrchestrator:
    def __init__(self, unified_retriever, answer_generator, citation_agent):
        self.retriever = unified_retriever
        self.answer_generator = answer_generator
        self.citation_agent = citation_agent

    def run(self, question, mode="both"):
        start = time.perf_counter()
        docs = self.retriever.retrieve(question, mode=mode)
        generation = self.answer_generator.generate(question, docs)
        citation_result = self.citation_agent.process(generation["answer"], docs)
        citation_check = verify_generated_citations(generation["answer"], docs)
        return {
            "question": question,
            "mode": mode,
            "answer": generation["answer"],
            "confidence": citation_result["confidence"],
            "max_similarity": citation_result["max_similarity"],
            "citations": citation_result["citations"],
            "citation_check": citation_check,
            "retrieved_documents": docs,
            "input_tokens": generation["input_tokens"],
            "output_tokens": generation["output_tokens"],
            "latency_seconds": round(time.perf_counter() - start, 3),
        }

print("✅ Unified official/user/both retrieval flow ready")


✅ Unified official/user/both retrieval flow ready


In [28]:
# Cell 20 — Upload a test PDF for user-document evaluation
from google.colab import files

uploaded = files.upload()
assert uploaded, "No PDF uploaded."
uploaded_name = next(iter(uploaded))
assert uploaded_name.lower().endswith(".pdf"), "Please upload a PDF."

USER_WORKSPACE_ID = "evaluation_user"
user_upload_agent = UserUploadAgent(USER_WORKSPACE_ID)
upload_result = user_upload_agent.add_pdf(uploaded_name)

print(upload_result)
print("User workspace:", USER_WORKSPACE_ID)
print("User DB path:", USER_UPLOAD_CHROMA_PATH / USER_WORKSPACE_ID)


Saving 20251107814584932.pdf to 20251107814584932.pdf
{'workspace_id': 'evaluation_user', 'filename': '20251107814584932.pdf', 'chunks_added': 33}
User workspace: evaluation_user
User DB path: /content/drive/MyDrive/Nyay_AI/user_upload_chroma/evaluation_user


In [29]:
# Cell 21 — User PDF Q&A test
unified_retriever = UnifiedRetrieverAgent(
    official_retriever, user_upload_agent, k=TOP_K
)
unified_orchestrator = UnifiedNyayAIOrchestrator(
    unified_retriever, answer_agent, citation_agent
)

user_question = input("Enter a question about the uploaded PDF: ").strip()
user_result = unified_orchestrator.run(user_question, mode="user")

print("\nANSWER:\n", user_result["answer"])
print("\nCONFIDENCE:", user_result["confidence"])
print("MAX SIMILARITY:", round(user_result["max_similarity"], 4))
print("LATENCY:", user_result["latency_seconds"], "sec")
print("\nSOURCES:")
for s in user_result["citations"]:
    print(s["source_id"], s["filename"], "page", s["page"], "sim", round(s["similarity"], 4))


Enter a question about the uploaded PDF: What is the purpose of the Shree Shanaishwar Devasthan Trust (Shingnapur) Act, 2018?

ANSWER:
 The purpose of **The Shree Shanaishwar Devasthan Trust (Shingnapur) Act, 2018** is to:

1. **Re-constitute the public trust** registered under the name *Shree Shanaishwar Devasthan* at Shingnapur, District Ahmednagar [S1], [S2].
2. **Provide for better administration** and management of the Shree Shanaishwar Temple by a committee under direct State Government supervision [S1], [S2], [S3], [S4].
3. **Enable the Trust to provide better facilities** to devotees [S3], [S4].
4. Address matters connected with or incidental to the temple's administration and governance [S1], [S2], [S3], [S4].

CONFIDENCE: Corpus-Grounded
MAX SIMILARITY: 0.7827
LATENCY: 25.795 sec

SOURCES:
S1 20251107814584932.pdf page 3 sim 0.7827
S2 20251107814584932 (1).pdf page 3 sim 0.7827
S3 20251107814584932.pdf page 3 sim 0.753
S4 20251107814584932 (1).pdf page 3 sim 0.753


In [30]:
# Cell 22 — Mixed-query demo (user PDF + official corpus)
mixed_question = input(
    "Enter a question that may need BOTH the uploaded PDF and Maharashtra law: "
).strip()

mixed_result = unified_orchestrator.run(mixed_question, mode="both")

print("\nANSWER:\n", mixed_result["answer"])
print("\nCONFIDENCE:", mixed_result["confidence"])
print("LATENCY:", mixed_result["latency_seconds"], "sec")
print("\nRETRIEVED SOURCES:")
for s in mixed_result["citations"]:
    print(
        s["source_id"], "|", s["retrieval_source"], "|",
        s["filename"], "| page", s["page"], "| sim", round(s["similarity"], 4)
    )


Enter a question that may need BOTH the uploaded PDF and Maharashtra law: What does Section 45 say about the Maharashtra Rent Control Act, 1999?

ANSWER:
 Insufficient Evidence

CONFIDENCE: Corpus-Grounded
LATENCY: 3.071 sec

RETRIEVED SOURCES:
S1 | official_corpus | The Maharashtra Rent Control Act, 1999.pdf | page 4 | sim 0.8723
S2 | official_corpus | The Maharashtra Rent Control Act, 1999.pdf | page 43 | sim 0.7662
S3 | official_corpus | The Maharashtra Rent Control Act, 1999.pdf | page 5 | sim 0.7609
S4 | official_corpus | The Maharashtra Rent Control Act, 1999.pdf | page 32 | sim 0.7547
S5 | official_corpus | Manual for Handling Legislation and Subordinate Legislation Work.pdf | page 179 | sim 0.7543


What is the purpose of the Shree Shanaishwar Devasthan Trust Act, 2018?
When did the Act come into force?
What is the appointed day under the Act?
Who constitutes the Management Committee?
What is the term of office of a committee member?
What are the qualifications for becoming a member?
What are the duties and powers of the Management Committee?
Who appoints the Executive Officer?
What is the maximum emergency expenditure the Executive Officer can authorize?
What is the purpose of the Trust Fund?
How is the Management Fund used?
How often must the accounts be audited?
What powers does the State Government have over the Committee?
What happens if the Management Committee is dissolved?
Does the Maharashtra Rent Control Act, 1999 apply to property of this Devasthan Trust?

In [31]:
# Cell 22 — Mixed-query demo (user PDF + official corpus)
mixed_question = input(
    "Enter a question that may need BOTH the uploaded PDF and Maharashtra law: "
).strip()

mixed_result = unified_orchestrator.run(mixed_question, mode="both")

print("\nANSWER:\n", mixed_result["answer"])
print("\nCONFIDENCE:", mixed_result["confidence"])
print("LATENCY:", mixed_result["latency_seconds"], "sec")
print("\nRETRIEVED SOURCES:")
for s in mixed_result["citations"]:
    print(
        s["source_id"], "|", s["retrieval_source"], "|",
        s["filename"], "| page", s["page"], "| sim", round(s["similarity"], 4)
    )


Enter a question that may need BOTH the uploaded PDF and Maharashtra law: Under what circumstances can the State Government dissolve or supersede the Management Committee?

ANSWER:
 Based on the retrieved legal sources, the circumstances depend on the specific body/act:

### 1. Devasthan Trust Committees (State Government)
The State Government may dissolve or supersede the committee for a period not exceeding six months **as it may deem fit**, provided it communicates the specific grounds in writing to the committee [S1], [S2].

### 2. Co-operative Society Committees
Supersession of an elected managing committee is an exception and can be resorted to **only in exceptional circumstances** [S5]. Specific grounds for supersession (under Section 78A) include [S3]:
* The committee or a member commits an act prejudicial to the interest of the society or its members [S3].
* The State Co-operative Election Authority fails to conduct elections in accordance with the Act [S3].
* A situation aris

In [32]:
# Cell 23 — Official evaluation set (30 questions)

OFFICIAL_EVAL_CSV = EVALUATION_PATH / "evaluation_questions.csv"

questions = [
    "What is the purpose of the Maharashtra Rent Control Act, 1999?",
    "What are the grounds for eviction under the Maharashtra Rent Control Act?",
    "What is meant by landlord under the Maharashtra Rent Control Act?",
    "What is meant by tenant under the Maharashtra Rent Control Act?",
    "What are the duties of a landlord under the Maharashtra Rent Control Act?",
    "What restrictions apply to rent increases under the Maharashtra Rent Control Act?",
    "What provisions deal with recovery of possession under the Maharashtra Rent Control Act?",
    "What establishments are covered by the Maharashtra Shops and Establishments Act, 2017?",
    "What are the registration requirements under the Maharashtra Shops and Establishments Act?",
    "What are the working-hour provisions under the Maharashtra Shops and Establishments Act?",
    "What provisions concern weekly holidays under the Maharashtra Shops and Establishments Act?",
    "What provisions concern leave under the Maharashtra Shops and Establishments Act?",
    "What is the purpose of the Maharashtra Land Revenue Code, 1966?",
    "What powers are given to revenue officers under the Maharashtra Land Revenue Code?",
    "What provisions concern assessment of land revenue?",
    "What provisions concern land records?",
    "What provisions concern government land under the Maharashtra Land Revenue Code?",
    "What is the purpose of the Maharashtra Police Act?",
    "What powers are provided for maintaining public order under the Maharashtra Police Act?",
    "What penalties are provided under the Maharashtra Police Act?",
    "What is the purpose of the Maharashtra Stamp Act?",
    "How is stamp duty determined?",
    "What happens when an instrument is insufficiently stamped?",
    "What is the purpose of the Real Estate (Regulation and Development) Act?",
    "What is the role of the real estate regulatory authority?",
    "What issue is discussed in a selected Bombay High Court judgment?",
    "What was the court's decision in the selected Bombay High Court judgment?",
    "What legal provision was considered by a selected Bombay High Court judgment?",
    "What is the subject of a selected Government Resolution?",
    "What direction or policy does a selected Government Resolution provide?"
]

# Create/recreate the 30-question evaluation file
official_eval = pd.DataFrame({
    "question_id": range(1, 31),
    "question": questions,
    "relevant_document_ids": [""] * 30,
    "relevant_pages": [""] * 30
})

official_eval.to_csv(OFFICIAL_EVAL_CSV, index=False)

print("Created official evaluation set:")
print(OFFICIAL_EVAL_CSV)
print("Number of questions:", len(official_eval))

assert len(official_eval) == 30

display(official_eval)

Created official evaluation set:
/content/drive/MyDrive/Nyay_AI/evaluation/evaluation_questions.csv
Number of questions: 30


,question_id,question,relevant_document_ids,relevant_pages
0,1,What is the purpose of the Maharashtra Rent Co...,,
1,2,What are the grounds for eviction under the Ma...,,
2,3,What is meant by landlord under the Maharashtr...,,
3,4,What is meant by tenant under the Maharashtra ...,,
4,5,What are the duties of a landlord under the Ma...,,
5,6,What restrictions apply to rent increases unde...,,
6,7,What provisions deal with recovery of possessi...,,
7,8,What establishments are covered by the Maharas...,,
8,9,What are the registration requirements under t...,,
9,10,What are the working-hour provisions under the...,,


In [33]:
# Cell 24 — Official retrieval evaluation
official_rows = []

for _, row in official_eval.iterrows():
    start = time.perf_counter()
    docs = official_retriever.retrieve(row["question"])
    latency = time.perf_counter() - start

    official_rows.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "retrieved_document_ids": ",".join(
            str(d.metadata.get("document_id", "")) for d in docs
        ),
        "retrieved_pages": ",".join(
            str(d.metadata.get("page", "")) for d in docs
        ),
        "max_similarity": max(
            [float(d.metadata.get("similarity", 0.0)) for d in docs],
            default=0.0
        ),
        "retrieval_latency_seconds": round(latency, 3),
    })

official_retrieval_df = pd.DataFrame(official_rows)
display(official_retrieval_df.head())
print("✅ Official retrieval evaluation completed")


,question_id,question,retrieved_document_ids,retrieved_pages,max_similarity,retrieval_latency_seconds
0,1,What is the purpose of the Maharashtra Rent Co...,"DOC_020,DOC_020,DOC_020,DOC_020,DOC_020","4,5,43,3,32",0.896250,0.321
1,2,What are the grounds for eviction under the Ma...,"DOC_020,DOC_064,DOC_020,DOC_020,DOC_015","4,6,23,44,36",0.805184,0.382
2,3,What is meant by landlord under the Maharashtr...,"DOC_020,DOC_020,DOC_020,DOC_002,DOC_015","4,7,5,179,15",0.819359,0.395
3,4,What is meant by tenant under the Maharashtra ...,"DOC_020,DOC_020,DOC_020,DOC_014,DOC_020","4,7,5,2,3",0.809427,0.304
4,5,What are the duties of a landlord under the Ma...,"DOC_020,DOC_002,DOC_020,DOC_020,DOC_020","4,200,5,7,3",0.790014,0.361


✅ Official retrieval evaluation completed


In [34]:
# Cell 25 — Official Precision@5 / Recall@5 with document-level ground truth
def normalize_list(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [x.strip() for x in str(value).split(",") if x.strip()]

p5_vals, r5_vals = [], []

for _, row in official_eval.iterrows():
    relevant = set(normalize_list(row.get("relevant_document_ids", "")))
    if not relevant:
        continue

    rr = official_retrieval_df[
        official_retrieval_df["question_id"] == row["question_id"]
    ].iloc[0]
    retrieved = set(normalize_list(rr["retrieved_document_ids"]))
    hits = len(relevant & retrieved)

    p5_vals.append(hits / TOP_K)
    r5_vals.append(hits / len(relevant))

precision_at_5_official = float(np.mean(p5_vals)) if p5_vals else None
recall_at_5_official = float(np.mean(r5_vals)) if r5_vals else None

print("Official Precision@5:", precision_at_5_official)
print("Official Recall@5:", recall_at_5_official)
print("Average max similarity:", round(official_retrieval_df["max_similarity"].mean(), 4))
print("Threshold pass rate:", round((official_retrieval_df["max_similarity"] >= CONFIDENCE_THRESHOLD).mean(), 4))
print("Average retrieval latency:", round(official_retrieval_df["retrieval_latency_seconds"].mean(), 3), "sec")


Official Precision@5: None
Official Recall@5: None
Average max similarity: 0.7364
Threshold pass rate: 0.6667
Average retrieval latency: 0.466 sec


In [35]:
# Cell 26 — Official 30-question Gemini answer evaluation

official_answer_rows = []

print("=" * 70)
print("OFFICIAL 30-QUESTION GEMINI ANSWER EVALUATION")
print("=" * 70)

for _, row in official_eval.iterrows():

    print(f"\nRunning Question {row['question_id']}...")

    try:
        result = official_orchestrator.run(row["question"])

        official_answer_rows.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "answer": result["answer"],
            "confidence": result["confidence"],
            "max_similarity": result["max_similarity"],
            "citation_format_accuracy": result["citation_check"]["citation_format_accuracy"],
            "input_tokens": result.get("input_tokens", 0),
            "output_tokens": result.get("output_tokens", 0),
            "total_tokens": (
                result.get("input_tokens", 0)
                + result.get("output_tokens", 0)
            ),
            "latency_seconds": result["latency_seconds"],
            "status": "Success",
            "error": "",
        })

        print(
            f"✅ Q{row['question_id']} "
            f"| Input: {result.get('input_tokens', 0)} "
            f"| Output: {result.get('output_tokens', 0)} "
            f"| Latency: {result['latency_seconds']:.2f}s"
        )

    except Exception as e:

        official_answer_rows.append({
            "question_id": row["question_id"],
            "question": row["question"],
            "answer": "",
            "confidence": "API Error",
            "max_similarity": 0.0,
            "citation_format_accuracy": 0.0,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "latency_seconds": None,
            "status": "Error",
            "error": str(e),
        })

        print(
            f"❌ Q{row['question_id']} ERROR: {str(e)}"
        )


# Create DataFrame
official_answers_df = pd.DataFrame(
    official_answer_rows
)


# Save Gemini evaluation results
official_answers_path = (
    EVALUATION_PATH
    / "official_30_gemini_answers.csv"
)

official_answers_df.to_csv(
    official_answers_path,
    index=False
)


# Summary
success_count = (
    official_answers_df["status"] == "Success"
).sum()

error_count = (
    official_answers_df["status"] == "Error"
).sum()


print("\n" + "=" * 70)
print("✅ GEMINI EVALUATION COMPLETED")
print("=" * 70)

print(
    "Total questions:",
    len(official_answers_df)
)

print(
    "Success:",
    success_count,
    "/",
    len(official_answers_df)
)

print(
    "Errors:",
    error_count
)

print(
    "Saved:",
    official_answers_path
)

display(official_answers_df)

OFFICIAL 30-QUESTION GEMINI ANSWER EVALUATION

Running Question 1...
✅ Q1 | Input: 1528 | Output: 107 | Latency: 30.09s

Running Question 2...
✅ Q2 | Input: 1935 | Output: 86 | Latency: 34.21s

Running Question 3...
✅ Q3 | Input: 2272 | Output: 2 | Latency: 29.68s

Running Question 4...
✅ Q4 | Input: 1657 | Output: 2 | Latency: 35.83s

Running Question 5...
✅ Q5 | Input: 1847 | Output: 2 | Latency: 43.48s

Running Question 6...
✅ Q6 | Input: 2211 | Output: 365 | Latency: 28.13s

Running Question 7...
✅ Q7 | Input: 2004 | Output: 224 | Latency: 32.88s

Running Question 8...
✅ Q8 | Input: 1654 | Output: 167 | Latency: 7.20s

Running Question 9...
✅ Q9 | Input: 2182 | Output: 487 | Latency: 39.90s

Running Question 10...
✅ Q10 | Input: 2136 | Output: 367 | Latency: 38.01s

Running Question 11...
✅ Q11 | Input: 2146 | Output: 255 | Latency: 37.28s

Running Question 12...
✅ Q12 | Input: 1643 | Output: 74 | Latency: 11.07s

Running Question 13...
✅ Q13 | Input: 2254 | Output: 46 | Latency: 3

,question_id,question,answer,confidence,max_similarity,citation_format_accuracy,input_tokens,output_tokens,total_tokens,latency_seconds,status,error
0,1,What is the purpose of the Maharashtra Rent Co...,The purpose of the Maharashtra Rent Control Ac...,Corpus-Grounded,0.896250,1.0,1528,107,1635,30.095,Success,
1,2,What are the grounds for eviction under the Ma...,Insufficient Evidence\n\nThe retrieved sources...,Corpus-Grounded,0.805184,1.0,1935,86,2021,34.211,Success,
2,3,What is meant by landlord under the Maharashtr...,Insufficient Evidence,Corpus-Grounded,0.819359,0.0,2272,2,2274,29.678,Success,
3,4,What is meant by tenant under the Maharashtra ...,Insufficient Evidence,Corpus-Grounded,0.809427,0.0,1657,2,1659,35.826,Success,
4,5,What are the duties of a landlord under the Ma...,Insufficient Evidence,Corpus-Grounded,0.790014,0.0,1847,2,1849,43.483,Success,
5,6,What restrictions apply to rent increases unde...,"Under the Maharashtra Rent Control Act, 1999, ...",Corpus-Grounded,0.788658,1.0,2211,365,2576,28.128,Success,
6,7,What provisions deal with recovery of possessi...,"Under the Maharashtra Rent Control Act, 1999, ...",Corpus-Grounded,0.810957,1.0,2004,224,2228,32.877,Success,
7,8,What establishments are covered by the Maharas...,Under the Maharashtra Shops and Establishments...,Corpus-Grounded,0.824088,1.0,1654,167,1821,7.196,Success,
8,9,What are the registration requirements under t...,Under the Maharashtra Shops and Establishments...,Corpus-Grounded,0.712657,1.0,2182,487,2669,39.899,Success,
9,10,What are the working-hour provisions under the...,Under the Maharashtra Shops and Establishments...,Corpus-Grounded,0.781600,1.0,2136,367,2503,38.007,Success,


In [43]:
# Resume Gemini evaluation — ONLY failed questions

import pandas as pd

results_path = EVALUATION_PATH / "official_30_gemini_answers.csv"

old_results = pd.read_csv(results_path)

failed_ids = old_results.loc[
    old_results["status"] == "Error",
    "question_id"
].tolist()

print("Already successful:",
      old_results.loc[old_results["status"] == "Success", "question_id"].tolist())

print("Questions to retry:", failed_ids)

Already successful: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Questions to retry: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]


In [44]:
# Resume Gemini evaluation — Q16 to Q30 only

import time
import pandas as pd

results_path = EVALUATION_PATH / "official_30_gemini_answers.csv"

old_results = pd.read_csv(results_path)

# Questions that previously failed
failed_ids = old_results.loc[
    old_results["status"] == "Error",
    "question_id"
].tolist()

remaining_eval = official_eval[
    official_eval["question_id"].isin(failed_ids)
].copy()

print("Running only:", failed_ids)
print("Total remaining:", len(remaining_eval))

new_rows = []

for _, row in remaining_eval.iterrows():

    qid = row["question_id"]
    question = row["question"]

    print(f"\nRunning Question {qid}...")

    try:
        start_time = time.perf_counter()

        result = official_orchestrator.run(question)

        latency = time.perf_counter() - start_time

        input_tokens = result.get("input_tokens", 0)
        output_tokens = result.get("output_tokens", 0)

        new_rows.append({
            "question_id": qid,
            "question": question,
            "answer": result["answer"],
            "confidence": result["confidence"],
            "max_similarity": result["max_similarity"],
            "citation_format_accuracy":
                result["citation_check"]["citation_format_accuracy"],
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens":
                input_tokens + output_tokens,
            "latency_seconds":
                result["latency_seconds"],
            "status": "Success",
            "error": ""
        })

        print(
            f"✅ Q{qid} | "
            f"Input: {input_tokens} | "
            f"Output: {output_tokens} | "
            f"Latency: {latency:.2f}s"
        )

    except Exception as e:

        new_rows.append({
            "question_id": qid,
            "question": question,
            "answer": "",
            "confidence": "API Error",
            "max_similarity": 0.0,
            "citation_format_accuracy": 0.0,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "latency_seconds": None,
            "status": "Error",
            "error": str(e)
        })

        print(f"❌ Q{qid}: {str(e)}")


# Replace old failed rows with new successful rows
new_rows_df = pd.DataFrame(new_rows)

successful_old = old_results[
    old_results["status"] == "Success"
].copy()

final_results = pd.concat(
    [successful_old, new_rows_df],
    ignore_index=True
)

# Sort Q1 → Q30
final_results = final_results.sort_values(
    "question_id"
).reset_index(drop=True)

# Save final combined file
final_results.to_csv(
    results_path,
    index=False
)

print("\n" + "=" * 70)
print("FINAL GEMINI EVALUATION FILE")
print("=" * 70)

print(
    "Success:",
    (final_results["status"] == "Success").sum(),
    "/ 30"
)

print(
    "Errors:",
    (final_results["status"] == "Error").sum()
)

print("Saved:", results_path)

display(final_results)

Running only: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Total remaining: 15

Running Question 16...
✅ Q16 | Input: 2404 | Output: 205 | Latency: 38.76s

Running Question 17...
✅ Q17 | Input: 2872 | Output: 258 | Latency: 35.12s

Running Question 18...
✅ Q18 | Input: 2606 | Output: 168 | Latency: 32.48s

Running Question 19...
Gemini temporarily unavailable. Retrying in 5s...
✅ Q19 | Input: 2783 | Output: 336 | Latency: 53.36s

Running Question 20...
Gemini temporarily unavailable. Retrying in 5s...
✅ Q20 | Input: 2832 | Output: 316 | Latency: 49.05s

Running Question 21...
✅ Q21 | Input: 2691 | Output: 55 | Latency: 30.40s

Running Question 22...
✅ Q22 | Input: 2602 | Output: 197 | Latency: 34.35s

Running Question 23...
✅ Q23 | Input: 2776 | Output: 232 | Latency: 36.40s

Running Question 24...
✅ Q24 | Input: 2306 | Output: 248 | Latency: 29.53s

Running Question 25...
✅ Q25 | Input: 2106 | Output: 248 | Latency: 33.54s

Running Question 26...
✅ Q26 | Input: 2332 | 

,question_id,question,answer,confidence,max_similarity,citation_format_accuracy,input_tokens,output_tokens,total_tokens,latency_seconds,status,error
0,1,What is the purpose of the Maharashtra Rent Co...,The purpose of the Maharashtra Rent Control Ac...,Corpus-Grounded,0.896250,1.0,1528,107,1635,30.095,Success,NaN
1,2,What are the grounds for eviction under the Ma...,Insufficient Evidence\n\nThe retrieved sources...,Corpus-Grounded,0.805184,1.0,1935,86,2021,34.211,Success,NaN
2,3,What is meant by landlord under the Maharashtr...,Insufficient Evidence,Corpus-Grounded,0.819359,0.0,2272,2,2274,29.678,Success,NaN
3,4,What is meant by tenant under the Maharashtra ...,Insufficient Evidence,Corpus-Grounded,0.809427,0.0,1657,2,1659,35.826,Success,NaN
4,5,What are the duties of a landlord under the Ma...,Insufficient Evidence,Corpus-Grounded,0.790014,0.0,1847,2,1849,43.483,Success,NaN
5,6,What restrictions apply to rent increases unde...,"Under the Maharashtra Rent Control Act, 1999, ...",Corpus-Grounded,0.788658,1.0,2211,365,2576,28.128,Success,NaN
6,7,What provisions deal with recovery of possessi...,"Under the Maharashtra Rent Control Act, 1999, ...",Corpus-Grounded,0.810957,1.0,2004,224,2228,32.877,Success,NaN
7,8,What establishments are covered by the Maharas...,Under the Maharashtra Shops and Establishments...,Corpus-Grounded,0.824088,1.0,1654,167,1821,7.196,Success,NaN
8,9,What are the registration requirements under t...,Under the Maharashtra Shops and Establishments...,Corpus-Grounded,0.712657,1.0,2182,487,2669,39.899,Success,NaN
9,10,What are the working-hour provisions under the...,Under the Maharashtra Shops and Establishments...,Corpus-Grounded,0.781600,1.0,2136,367,2503,38.007,Success,NaN


In [49]:
# Cell 27 — Official 30-Question Retrieval Evaluation
# IMPORTANT: Do NOT use Gemini in this cell.

import pandas as pd
import time

retrieval_rows = []

print("=" * 70)
print("OFFICIAL 30-QUESTION RETRIEVAL EVALUATION")
print("=" * 70)

# Get RetrieverAgent from the existing official orchestrator
if hasattr(official_orchestrator, "retriever"):
    retrieval_agent = official_orchestrator.retriever
elif hasattr(official_orchestrator, "retriever_agent"):
    retrieval_agent = official_orchestrator.retriever_agent
elif hasattr(official_orchestrator, "retrieval_agent"):
    retrieval_agent = official_orchestrator.retrieval_agent
else:
    raise AttributeError(
        "RetrieverAgent not found inside official_orchestrator."
    )

print("Retriever found:", type(retrieval_agent))


for _, row in official_eval.iterrows():

    qid = row["question_id"]
    question = row["question"]

    print(f"\nRunning retrieval Q{qid}...")

    start_time = time.perf_counter()

    try:
        # IMPORTANT:
        # Your RetrieverAgent.retrieve() does NOT accept k=5.
        # Therefore call it with only the question.
        retrieved_docs = retrieval_agent.retrieve(question)

        latency = time.perf_counter() - start_time

        # Make sure we evaluate only Top-5
        retrieved_docs = retrieved_docs[:5]

        retrieved_ids = []
        retrieved_pages = []
        similarities = []

        for item in retrieved_docs:

            # LangChain Document
            if hasattr(item, "metadata"):
                metadata = item.metadata or {}

            # Dictionary result
            elif isinstance(item, dict):
                metadata = item.get("metadata", {}) or {}

            else:
                metadata = {}

            # Document ID
            document_id = (
                metadata.get("document_id")
                or metadata.get("doc_id")
                or metadata.get("source")
                or metadata.get("file_name")
                or ""
            )

            # Page number
            page = (
                metadata.get("page")
                or metadata.get("page_number")
                or ""
            )

            # Similarity score
            similarity = (
                metadata.get("similarity")
                if metadata.get("similarity") is not None
                else metadata.get("score")
            )

            if similarity is None:
                similarity = 0.0

            try:
                similarity = float(similarity)
            except:
                similarity = 0.0

            retrieved_ids.append(str(document_id))
            retrieved_pages.append(str(page))
            similarities.append(similarity)

        # ------------------------------------------------------
        # Ground-truth relevant documents
        # ------------------------------------------------------

        relevant_raw = row.get(
            "relevant_document_ids",
            ""
        )

        if pd.isna(relevant_raw):
            relevant_raw = ""

        relevant_ids = [
            x.strip()
            for x in str(relevant_raw).split(";")
            if x.strip()
        ]

        # ------------------------------------------------------
        # Precision@5
        # ------------------------------------------------------

        relevant_retrieved = sum(
            1
            for doc_id in retrieved_ids
            if doc_id in relevant_ids
        )

        precision_at_5 = relevant_retrieved / 5

        # ------------------------------------------------------
        # Recall@5
        # ------------------------------------------------------

        if len(relevant_ids) > 0:
            recall_at_5 = (
                relevant_retrieved / len(relevant_ids)
            )
        else:
            recall_at_5 = 0.0

        # ------------------------------------------------------
        # Maximum similarity
        # ------------------------------------------------------

        max_similarity = (
            max(similarities)
            if similarities
            else 0.0
        )

        retrieval_rows.append({
            "question_id": qid,
            "question": question,
            "retrieved_document_ids": ";".join(
                retrieved_ids
            ),
            "retrieved_pages": ";".join(
                retrieved_pages
            ),
            "relevant_document_ids": ";".join(
                relevant_ids
            ),
            "precision_at_5": round(
                precision_at_5, 4
            ),
            "recall_at_5": round(
                recall_at_5, 4
            ),
            "max_similarity": round(
                max_similarity, 4
            ),
            "latency_seconds": round(
                latency, 4
            ),
            "status": "Success"
        })

        print(
            f"✅ Q{qid} | "
            f"P@5={precision_at_5:.2f} | "
            f"R@5={recall_at_5:.2f} | "
            f"Similarity={max_similarity:.4f}"
        )

    except Exception as e:

        retrieval_rows.append({
            "question_id": qid,
            "question": question,
            "retrieved_document_ids": "",
            "retrieved_pages": "",
            "relevant_document_ids": str(
                row.get(
                    "relevant_document_ids",
                    ""
                )
            ),
            "precision_at_5": 0.0,
            "recall_at_5": 0.0,
            "max_similarity": 0.0,
            "latency_seconds": None,
            "status": "Error",
            "error": str(e)
        })

        print(f"❌ Q{qid}: {e}")


# ----------------------------------------------------------
# Save results
# ----------------------------------------------------------

retrieval_df = pd.DataFrame(
    retrieval_rows
)

retrieval_path = (
    EVALUATION_PATH
    / "official_30_retrieval_results.csv"
)

retrieval_df.to_csv(
    retrieval_path,
    index=False
)

print("\n" + "=" * 70)
print("RETRIEVAL EVALUATION COMPLETED")
print("=" * 70)

print("Questions:", len(retrieval_df))

print(
    "Successful:",
    (
        retrieval_df["status"] == "Success"
    ).sum()
)

print(
    "Errors:",
    (
        retrieval_df["status"] == "Error"
    ).sum()
)

print("Saved:", retrieval_path)

display(retrieval_df)

OFFICIAL 30-QUESTION RETRIEVAL EVALUATION
Retriever found: <class '__main__.RetrieverAgent'>

Running retrieval Q1...
✅ Q1 | P@5=0.00 | R@5=0.00 | Similarity=0.8963

Running retrieval Q2...
✅ Q2 | P@5=0.00 | R@5=0.00 | Similarity=0.8052

Running retrieval Q3...
✅ Q3 | P@5=0.00 | R@5=0.00 | Similarity=0.8194

Running retrieval Q4...
✅ Q4 | P@5=0.00 | R@5=0.00 | Similarity=0.8094

Running retrieval Q5...
✅ Q5 | P@5=0.00 | R@5=0.00 | Similarity=0.7900

Running retrieval Q6...
✅ Q6 | P@5=0.00 | R@5=0.00 | Similarity=0.7887

Running retrieval Q7...
✅ Q7 | P@5=0.00 | R@5=0.00 | Similarity=0.8110

Running retrieval Q8...
✅ Q8 | P@5=0.00 | R@5=0.00 | Similarity=0.8241

Running retrieval Q9...
✅ Q9 | P@5=0.00 | R@5=0.00 | Similarity=0.7127

Running retrieval Q10...
✅ Q10 | P@5=0.00 | R@5=0.00 | Similarity=0.7816

Running retrieval Q11...
✅ Q11 | P@5=0.00 | R@5=0.00 | Similarity=0.6992

Running retrieval Q12...
✅ Q12 | P@5=0.00 | R@5=0.00 | Similarity=0.7832

Running retrieval Q13...
✅ Q13 | P@5

,question_id,question,retrieved_document_ids,retrieved_pages,relevant_document_ids,precision_at_5,recall_at_5,max_similarity,latency_seconds,status
0,1,What is the purpose of the Maharashtra Rent Co...,DOC_020;DOC_020;DOC_020;DOC_020;DOC_020,4;5;43;3;32,,0.0,0.0,0.8963,1.0805,Success
1,2,What are the grounds for eviction under the Ma...,DOC_020;DOC_064;DOC_020;DOC_020;DOC_015,4;6;23;44;36,,0.0,0.0,0.8052,0.7813,Success
2,3,What is meant by landlord under the Maharashtr...,DOC_020;DOC_020;DOC_020;DOC_002;DOC_015,4;7;5;179;15,,0.0,0.0,0.8194,0.5920,Success
3,4,What is meant by tenant under the Maharashtra ...,DOC_020;DOC_020;DOC_020;DOC_014;DOC_020,4;7;5;2;3,,0.0,0.0,0.8094,0.4532,Success
4,5,What are the duties of a landlord under the Ma...,DOC_020;DOC_002;DOC_020;DOC_020;DOC_020,4;200;5;7;3,,0.0,0.0,0.7900,0.5665,Success
5,6,What restrictions apply to rent increases unde...,DOC_020;DOC_020;DOC_020;DOC_020;DOC_020,4;13;1;11;7,,0.0,0.0,0.7887,0.5928,Success
6,7,What provisions deal with recovery of possessi...,DOC_020;DOC_002;DOC_020;DOC_020;DOC_020,2;179;4;44;5,,0.0,0.0,0.8110,0.5662,Success
7,8,What establishments are covered by the Maharas...,DOC_013;DOC_013;DOC_013;DOC_013;DOC_013,4;5;2;1;18,,0.0,0.0,0.8241,0.5394,Success
8,9,What are the registration requirements under t...,DOC_013;DOC_013;DOC_013;DOC_013;DOC_013,8;2;9;1;8,,0.0,0.0,0.7127,0.6557,Success
9,10,What are the working-hour provisions under the...,DOC_013;DOC_013;DOC_013;DOC_010;DOC_013,10;1;4;68;5,,0.0,0.0,0.7816,0.5920,Success


In [50]:
# Cell 28 — Retrieval Evaluation Summary

import pandas as pd

# Load retrieval results
retrieval_path = (
    EVALUATION_PATH
    / "official_30_retrieval_results.csv"
)

retrieval_df = pd.read_csv(retrieval_path)

# Keep only successful runs
successful = retrieval_df[
    retrieval_df["status"] == "Success"
].copy()

# Calculate averages
mean_precision = successful["precision_at_5"].mean()
mean_recall = successful["recall_at_5"].mean()
mean_similarity = successful["max_similarity"].mean()
mean_latency = successful["latency_seconds"].mean()

# Summary
retrieval_summary = pd.DataFrame({
    "Metric": [
        "Questions Evaluated",
        "Successful Retrievals",
        "Retrieval Errors",
        "Mean Precision@5",
        "Mean Recall@5",
        "Mean Maximum Similarity",
        "Mean Retrieval Latency (seconds)"
    ],
    "Value": [
        len(retrieval_df),
        len(successful),
        (retrieval_df["status"] == "Error").sum(),
        round(mean_precision, 4),
        round(mean_recall, 4),
        round(mean_similarity, 4),
        round(mean_latency, 4)
    ]
})

print("=" * 60)
print("RETRIEVAL EVALUATION SUMMARY")
print("=" * 60)

display(retrieval_summary)

print("\nNote:")
print("P@5 and R@5 are currently 0 because the")
print("'relevant_document_ids' ground-truth field is empty.")
print("Do NOT report these P@5/R@5 values as final research results.")

RETRIEVAL EVALUATION SUMMARY


,Metric,Value
0,Questions Evaluated,30.0000
1,Successful Retrievals,30.0000
2,Retrieval Errors,0.0000
3,Mean Precision@5,0.0000
4,Mean Recall@5,0.0000
5,Mean Maximum Similarity,0.7364
6,Mean Retrieval Latency (seconds),0.7362



Note:
P@5 and R@5 are currently 0 because the
'relevant_document_ids' ground-truth field is empty.
Do NOT report these P@5/R@5 values as final research results.


In [51]:
# Cell 29 — Check Official Evaluation Ground Truth

import pandas as pd

print("=" * 70)
print("CHECKING OFFICIAL EVALUATION GROUND TRUTH")
print("=" * 70)

# Load the evaluation questions
eval_path = EVALUATION_PATH / "evaluation_questions.csv"

official_eval_check = pd.read_csv(eval_path)

print("Total questions:", len(official_eval_check))
print()

# Show important columns
print("Columns:")
for col in official_eval_check.columns:
    print(" -", col)

print("\n" + "=" * 70)
print("GROUND-TRUTH DATA")
print("=" * 70)

display(
    official_eval_check[
        [
            "question_id",
            "question",
            "relevant_document_ids",
            "relevant_pages"
        ]
    ]
)

# Count missing ground truth
missing_doc_ids = (
    official_eval_check["relevant_document_ids"]
    .isna()
    .sum()
)

missing_pages = (
    official_eval_check["relevant_pages"]
    .isna()
    .sum()
)

print("\n" + "=" * 70)
print("GROUND-TRUTH STATUS")
print("=" * 70)

print(
    "Missing relevant_document_ids:",
    missing_doc_ids,
    "of",
    len(official_eval_check)
)

print(
    "Missing relevant_pages:",
    missing_pages,
    "of",
    len(official_eval_check)
)

if missing_doc_ids == len(official_eval_check):
    print("\n⚠️ All document ground truths are empty.")
    print("P@5 and R@5 cannot be calculated yet.")

elif missing_doc_ids > 0:
    print("\n⚠️ Some document ground truths are missing.")

else:
    print("\n✅ Document ground truth is available.")

CHECKING OFFICIAL EVALUATION GROUND TRUTH
Total questions: 30

Columns:
 - question_id
 - question
 - relevant_document_ids
 - relevant_pages

GROUND-TRUTH DATA


,question_id,question,relevant_document_ids,relevant_pages
0,1,What is the purpose of the Maharashtra Rent Co...,NaN,NaN
1,2,What are the grounds for eviction under the Ma...,NaN,NaN
2,3,What is meant by landlord under the Maharashtr...,NaN,NaN
3,4,What is meant by tenant under the Maharashtra ...,NaN,NaN
4,5,What are the duties of a landlord under the Ma...,NaN,NaN
5,6,What restrictions apply to rent increases unde...,NaN,NaN
6,7,What provisions deal with recovery of possessi...,NaN,NaN
7,8,What establishments are covered by the Maharas...,NaN,NaN
8,9,What are the registration requirements under t...,NaN,NaN
9,10,What are the working-hour provisions under the...,NaN,NaN



GROUND-TRUTH STATUS
Missing relevant_document_ids: 30 of 30
Missing relevant_pages: 30 of 30

⚠️ All document ground truths are empty.
P@5 and R@5 cannot be calculated yet.


In [52]:
# Cell 30 — Fill Verified Ground-Truth Document IDs
# No Gemini/API call used.

import pandas as pd

eval_path = EVALUATION_PATH / "evaluation_questions.csv"

df = pd.read_csv(eval_path)

# Ground-truth document mapping based on the official corpus structure
# DOC_020 = Maharashtra Rent Control Act
# DOC_013 = Maharashtra Shops and Establishments Act
# DOC_024 = Maharashtra Land Revenue Code
# DOC_021 = Maharashtra Police Act
# DOC_019 = Maharashtra Stamp Act
# DOC_028 = RERA-related law

ground_truth = {
    1:  ("DOC_020", ""),
    2:  ("DOC_020", ""),
    3:  ("DOC_020", ""),
    4:  ("DOC_020", ""),
    5:  ("DOC_020", ""),
    6:  ("DOC_020", ""),
    7:  ("DOC_020", ""),

    8:  ("DOC_013", ""),
    9:  ("DOC_013", ""),
    10: ("DOC_013", ""),
    11: ("DOC_013", ""),
    12: ("DOC_013", ""),

    13: ("DOC_024", ""),
    14: ("DOC_024", ""),
    15: ("DOC_024", ""),
    16: ("DOC_024", ""),
    17: ("DOC_024", ""),

    18: ("DOC_021", ""),
    19: ("DOC_021", ""),
    20: ("DOC_021", ""),

    21: ("DOC_019", ""),
    22: ("DOC_019", ""),
    23: ("DOC_019", ""),

    24: ("DOC_028", ""),
    25: ("DOC_028", ""),

    # These require identification of the exact selected judgment/GR.
    26: ("", ""),
    27: ("", ""),
    28: ("", ""),
    29: ("", ""),
    30: ("", "")
}

# Fill document IDs
df["relevant_document_ids"] = df["question_id"].map(
    lambda x: ground_truth.get(int(x), ("", ""))[0]
)

# Fill pages only where we have verified pages.
# Leave blank rather than inventing page numbers.
df["relevant_pages"] = df["question_id"].map(
    lambda x: ground_truth.get(int(x), ("", ""))[1]
)

# Save updated evaluation file
df.to_csv(
    eval_path,
    index=False
)

print("=" * 70)
print("GROUND-TRUTH DOCUMENT MAPPING")
print("=" * 70)

display(
    df[
        [
            "question_id",
            "question",
            "relevant_document_ids",
            "relevant_pages"
        ]
    ]
)

print("\nSaved:", eval_path)

print("\nDocument ground truth filled:")
print(
    df["relevant_document_ids"]
    .astype(str)
    .str.strip()
    .ne("")
    .sum(),
    "of",
    len(df)
)

print("\nQuestions still requiring exact source identification:")
print("Q26, Q27, Q28, Q29, Q30")

print("\n⚠️ Pages are intentionally left blank.")
print("Do NOT invent page numbers.")

GROUND-TRUTH DOCUMENT MAPPING


,question_id,question,relevant_document_ids,relevant_pages
0,1,What is the purpose of the Maharashtra Rent Co...,DOC_020,
1,2,What are the grounds for eviction under the Ma...,DOC_020,
2,3,What is meant by landlord under the Maharashtr...,DOC_020,
3,4,What is meant by tenant under the Maharashtra ...,DOC_020,
4,5,What are the duties of a landlord under the Ma...,DOC_020,
5,6,What restrictions apply to rent increases unde...,DOC_020,
6,7,What provisions deal with recovery of possessi...,DOC_020,
7,8,What establishments are covered by the Maharas...,DOC_013,
8,9,What are the registration requirements under t...,DOC_013,
9,10,What are the working-hour provisions under the...,DOC_013,



Saved: /content/drive/MyDrive/Nyay_AI/evaluation/evaluation_questions.csv

Document ground truth filled:
25 of 30

Questions still requiring exact source identification:
Q26, Q27, Q28, Q29, Q30

⚠️ Pages are intentionally left blank.
Do NOT invent page numbers.


In [54]:
# Cell 31A — Find the dataset folder

from pathlib import Path

print("=" * 70)
print("SEARCHING FOR DATASET")
print("=" * 70)

possible_paths = [
    Path("/content/drive/MyDrive/datset 2"),
    Path("/content/drive/MyDrive/dataset 2"),
    Path("/content/drive/MyDrive/Nyay_AI"),
    Path("/content/drive/MyDrive"),
]

for path in possible_paths:
    print(f"\nPath: {path}")
    print("Exists:", path.exists())

    if path.exists():
        pdfs = list(path.rglob("*.pdf"))
        print("PDFs found:", len(pdfs))

        if pdfs:
            print("First few PDFs:")
            for pdf in pdfs[:10]:
                print(" -", pdf)

SEARCHING FOR DATASET

Path: /content/drive/MyDrive/datset 2
Exists: True
PDFs found: 65
First few PDFs:
 - /content/drive/MyDrive/datset 2/Circulars and Government Resolutions issued by Department/Manual for Handling Legislation and Subordinate Legislation Work.pdf
 - /content/drive/MyDrive/datset 2/Circulars and Government Resolutions issued by Department/Checklists for CriminalCivil Proposals.pdf
 - /content/drive/MyDrive/datset 2/Circulars and Government Resolutions issued by Department/Procedure in case reference made to the Remembrancer of Legal Affairs.pdf
 - /content/drive/MyDrive/datset 2/Circulars and Government Resolutions issued by Department/Rules of procedure for dealing of Legislation work (Bills  Ordinances) and Subordinate Legislation work.pdf
 - /content/drive/MyDrive/datset 2/judgments/bombay_high_court/Slum/MN_Koli_WP_690_2024.pdf
 - /content/drive/MyDrive/datset 2/judgments/bombay_high_court/MRTP/Baburao_Gangadhar_Jadhav_MRTP.pdf
 - /content/drive/MyDrive/datset 2/

In [56]:
# Cell 32 — List all Judgment and GR documents

from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

print("=" * 70)
print("BOMBAY HIGH COURT JUDGMENTS")
print("=" * 70)

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

print("Total judgments:", len(judgment_files))
print()

for i, pdf in enumerate(judgment_files, 1):
    print(f"{i:02d}. {pdf.relative_to(DATASET_DIR)}")


print("\n" + "=" * 70)
print("GOVERNMENT RESOLUTIONS / CIRCULARS")
print("=" * 70)

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)

print("Total GR/Circular documents:", len(gr_files))
print()

for i, pdf in enumerate(gr_files, 1):
    print(f"{i:02d}. {pdf.relative_to(DATASET_DIR)}")

BOMBAY HIGH COURT JUDGMENTS
Total judgments: 19

01. judgments/bombay_high_court/Cooperative/Harish_Arora_WP_3433_2025.pdf
02. judgments/bombay_high_court/Cooperative/Uday_Dalal_WP_15089_2025.pdf
03. judgments/bombay_high_court/Labour/Best_Workers_Union_WP_1457_2015.pdf
04. judgments/bombay_high_court/Labour/Indo_Count_WP_4630_2025.pdf
05. judgments/bombay_high_court/Land Revenue/Government_Land_WPL_33728_2022.pdf
06. judgments/bombay_high_court/Land Revenue/Nirlon_Limited_WP_5398_2025.pdf
07. judgments/bombay_high_court/Land Revenue/Pravin_Gajanan_Thakur_WP_5716_2024.pdf
08. judgments/bombay_high_court/MOFA/Asian_Chemical_Industries_WP_11164_2024.pdf
09. judgments/bombay_high_court/MOFA/Shivranjan_Towers_WP_11281_2025.pdf
10. judgments/bombay_high_court/MRTP/Baburao_Gangadhar_Jadhav_MRTP.pdf
11. judgments/bombay_high_court/MRTP/Leenaben_MRTP.pdf
12. judgments/bombay_high_court/Maharashtra Rent Control Act/CRA_175_2025.pdf
13. judgments/bombay_high_court/Maharashtra Rent Control Act/CR

In [57]:
# Cell 33 — Find likely source documents for Q26–Q30
# No Gemini/API call.

import fitz
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)


def get_pdf_text(pdf_path):
    text = ""

    try:
        doc = fitz.open(pdf_path)

        for page in doc:
            text += page.get_text() + "\n"

        doc.close()

    except Exception as e:
        print("Error:", pdf_path.name, e)

    return text


# ----------------------------------------------------------
# Search judgment PDFs
# ----------------------------------------------------------

print("=" * 70)
print("JUDGMENT CONTENT SEARCH")
print("=" * 70)

keywords = [
    "decision",
    "judgment",
    "petition",
    "Bombay High Court",
    "section",
    "held"
]

for pdf in judgment_files:

    text = get_pdf_text(pdf)

    # Print basic information only
    print(
        f"\n{pdf.name}"
        f"\nCharacters: {len(text):,}"
    )

    # First useful portion of text
    clean_text = " ".join(text.split())

    print(
        "Preview:",
        clean_text[:500]
    )


# ----------------------------------------------------------
# Search GR / Circular PDFs
# ----------------------------------------------------------

print("\n" + "=" * 70)
print("GR / CIRCULAR CONTENT SEARCH")
print("=" * 70)

for pdf in gr_files:

    text = get_pdf_text(pdf)

    print(
        f"\n{pdf.name}"
        f"\nCharacters: {len(text):,}"
    )

    clean_text = " ".join(text.split())

    print(
        "Preview:",
        clean_text[:500]
    )

JUDGMENT CONTENT SEARCH

Harish_Arora_WP_3433_2025.pdf
Characters: 112,622
Preview: wp3433-2025-J Final.doc AGK IN THE HIGH COURT OF JUDICATURE AT BOMBAY CIVIL APPELLATE JURISDICTION WRIT PETITION NO.3433 OF 2025 1. Harish Arora, Age 45 years, Occupation: Business, residing at Flat Nos.1 & 2, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 2. Venetia Palia Age 50 years, Occupation: Business, residing at Flat No.3, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 3. Phiroze Karmali, Age 55 years, Occupation: Business, residing at Flat No.51, 

Uday_Dalal_WP_15089_2025.pdf
Characters: 52,554
Preview: wpl15089 & 15091-2025-J.doc AGK IN THE HIGH COURT OF JUDICATURE AT BOMBAY CIVIL APPELLATE JURISDICTION WRIT PETITION NO.15089 OF 2025 1. Uday Dalal, Flat No.2, 2nd Floor, Malboro House Coop. Housing Society Limited, 3-D, Dr. Gopalrao Deshmukh Marg, Peddar Road, Mumbai 400 026 2. Ajay Biyani, Flat No.4, 1st Floor, Malboro House Coop. Housing Society Limite

In [58]:
# Cell 33 — Find likely source documents for Q26–Q30
# Uses PyMuPDF
# No Gemini/API call

import pymupdf
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)


def get_pdf_text(pdf_path):
    text = ""

    try:
        doc = pymupdf.open(pdf_path)

        for page in doc:
            text += page.get_text() + "\n"

        doc.close()

    except Exception as e:
        print("Error:", pdf_path.name, e)

    return text


# ----------------------------------------------------------
# Bombay High Court Judgments
# ----------------------------------------------------------

print("=" * 70)
print("BOMBAY HIGH COURT JUDGMENTS")
print("=" * 70)

for i, pdf in enumerate(judgment_files, 1):

    text = get_pdf_text(pdf)

    clean_text = " ".join(text.split())

    print(f"\n{i:02d}. {pdf.name}")
    print(f"Characters: {len(text):,}")
    print(f"Preview: {clean_text[:500]}")


# ----------------------------------------------------------
# Government Resolutions / Circulars
# ----------------------------------------------------------

print("\n" + "=" * 70)
print("GOVERNMENT RESOLUTIONS / CIRCULARS")
print("=" * 70)

for i, pdf in enumerate(gr_files, 1):

    text = get_pdf_text(pdf)

    clean_text = " ".join(text.split())

    print(f"\n{i:02d}. {pdf.name}")
    print(f"Characters: {len(text):,}")
    print(f"Preview: {clean_text[:500]}")

BOMBAY HIGH COURT JUDGMENTS

01. Harish_Arora_WP_3433_2025.pdf
Characters: 112,622
Preview: wp3433-2025-J Final.doc AGK IN THE HIGH COURT OF JUDICATURE AT BOMBAY CIVIL APPELLATE JURISDICTION WRIT PETITION NO.3433 OF 2025 1. Harish Arora, Age 45 years, Occupation: Business, residing at Flat Nos.1 & 2, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 2. Venetia Palia Age 50 years, Occupation: Business, residing at Flat No.3, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 3. Phiroze Karmali, Age 55 years, Occupation: Business, residing at Flat No.51, 

02. Uday_Dalal_WP_15089_2025.pdf
Characters: 52,554
Preview: wpl15089 & 15091-2025-J.doc AGK IN THE HIGH COURT OF JUDICATURE AT BOMBAY CIVIL APPELLATE JURISDICTION WRIT PETITION NO.15089 OF 2025 1. Uday Dalal, Flat No.2, 2nd Floor, Malboro House Coop. Housing Society Limited, 3-D, Dr. Gopalrao Deshmukh Marg, Peddar Road, Mumbai 400 026 2. Ajay Biyani, Flat No.4, 1st Floor, Malboro House Coop. Housing So

In [60]:
# Cell 34 — Search Judgment / GR PDFs for Q26–Q30 topics
# Uses PyMuPDF
# No Gemini/API call

import pandas as pd
import pymupdf
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")


# ----------------------------------------------------------
# Files
# ----------------------------------------------------------

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)


# ----------------------------------------------------------
# PDF text reader
# ----------------------------------------------------------

def read_pdf(pdf_path):

    pages = []

    doc = pymupdf.open(pdf_path)

    for page_number, page in enumerate(doc, start=1):

        text = page.get_text()

        pages.append({
            "page": page_number,
            "text": text
        })

    doc.close()

    return pages


# ----------------------------------------------------------
# Search function
# ----------------------------------------------------------

def search_pdfs(files, keywords):

    results = []

    for pdf in files:

        try:
            pages = read_pdf(pdf)

            for page_data in pages:

                text = page_data["text"]

                text_lower = text.lower()

                matched = [
                    keyword
                    for keyword in keywords
                    if keyword.lower() in text_lower
                ]

                if matched:

                    results.append({
                        "file": pdf.name,
                        "path": str(pdf),
                        "page": page_data["page"],
                        "matched_keywords": ", ".join(matched),
                        "text_preview":
                            " ".join(text.split())[:700]
                    })

        except Exception as e:

            print(
                f"Error reading {pdf.name}: {e}"
            )

    return results


# ----------------------------------------------------------
# Q26–Q28 — Bombay High Court
# ----------------------------------------------------------

print("=" * 70)
print("SEARCHING BOMBAY HIGH COURT JUDGMENTS")
print("=" * 70)

judgment_keywords = [
    "Bombay High Court",
    "Petitioner",
    "Respondent",
    "Writ Petition",
    "Civil Revision Application",
    "judgment",
    "order"
]

judgment_results = search_pdfs(
    judgment_files,
    judgment_keywords
)

judgment_df = pd.DataFrame(
    judgment_results
)

print(
    "Matching judgment pages:",
    len(judgment_df)
)

display(judgment_df.head(50))


# ----------------------------------------------------------
# Q29–Q30 — Government Resolution / Circular
# ----------------------------------------------------------

print("\n" + "=" * 70)
print("SEARCHING GOVERNMENT RESOLUTIONS / CIRCULARS")
print("=" * 70)

gr_keywords = [
    "Government Resolution",
    "G.R.",
    "Government of Maharashtra",
    "Circular",
    "Resolution",
    "Department"
]

gr_results = search_pdfs(
    gr_files,
    gr_keywords
)

gr_df = pd.DataFrame(
    gr_results
)

print(
    "Matching GR/Circular pages:",
    len(gr_df)
)

display(gr_df.head(50))

SEARCHING BOMBAY HIGH COURT JUDGMENTS
Matching judgment pages: 406


,file,path,page,matched_keywords,text_preview
0,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,1,"Petitioner, Writ Petition",wp3433-2025-J Final.doc AGK IN THE HIGH COURT ...
1,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,3,Respondent,"wp3433-2025-J Final.doc Trishul Building, 18-A..."
2,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,4,"Petitioner, Respondent, judgment, order",wp3433-2025-J Final.doc Mr. Mayur Khandeparkar...
3,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,5,"Petitioner, Respondent, order",wp3433-2025-J Final.doc respondent No.10 on 20...
4,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,6,Respondent,wp3433-2025-J Final.doc constituted committee ...
5,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,7,"Petitioner, Respondent, order","wp3433-2025-J Final.doc complainants, the said..."
6,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,8,"Petitioner, Respondent, order",wp3433-2025-J Final.doc violated binding direc...
7,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,9,"Petitioner, Respondent, Writ Petition, judgmen...",wp3433-2025-J Final.doc further prejudice to t...
8,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,10,"Petitioner, Respondent, order",wp3433-2025-J Final.doc made in paragraph 19 t...
9,Harish_Arora_WP_3433_2025.pdf,/content/drive/MyDrive/datset 2/judgments/bomb...,11,"Respondent, order",wp3433-2025-J Final.doc Builders and Developer...



SEARCHING GOVERNMENT RESOLUTIONS / CIRCULARS
Matching GR/Circular pages: 163


,file,path,page,matched_keywords,text_preview
0,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,1,Department,M 32 i MANUAL FOR HANDLING LEGISLATION AND SUB...
1,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,3,"Government of Maharashtra, Department",M 32 iii PREFACE Law is a set of rules that ar...
2,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,4,"Government of Maharashtra, Department",M 32 iv 6. I hope this Manual will serve its b...
3,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,5,"Government of Maharashtra, Department","M 32 v DECLARATION I hereby declare that, this..."
4,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,11,Department,M 32 MANUAL FOR HANDALING LEGISLATION AND SUBO...
5,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,12,Department,M 32 x (B) Duties and responsibilities of offi...
6,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,13,Department,M 32 xi CHAPTER XII 12 General guidelines for ...
7,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,14,"Government Resolution, Resolution",M 32 xii CHAPTER XVII 17 Guidelines for drafti...
8,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,17,Department,M 32 xv (4) Right to services Notification.171...
9,Manual for Handling Legislation and Subordinat...,/content/drive/MyDrive/datset 2/Circulars and ...,20,Resolution,M 32 xviii Annexure XX Council Bill Citations....


In [61]:
# Cell 34 — Identify Candidate Sources for Q26–Q30
# PyMuPDF only — no Gemini

import pymupdf
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)


def get_pages(pdf_path):
    doc = pymupdf.open(pdf_path)
    pages = []

    for page_no, page in enumerate(doc, start=1):
        text = page.get_text()
        pages.append((page_no, text))

    doc.close()
    return pages


# ==========================================================
# JUDGMENT DOCUMENT IDENTIFICATION
# ==========================================================

print("=" * 70)
print("JUDGMENT DOCUMENTS")
print("=" * 70)

judgment_summary = []

for pdf in judgment_files:

    pages = get_pages(pdf)

    full_text = " ".join(
        text for _, text in pages
    )

    clean = " ".join(
        full_text.split()
    )

    # Look for useful identifying information
    terms = [
        "Writ Petition",
        "Civil Revision Application",
        "Petitioner",
        "Respondent",
        "High Court",
        "Maharashtra",
        "Section"
    ]

    matched = [
        term for term in terms
        if term.lower() in clean.lower()
    ]

    judgment_summary.append({
        "file": pdf.name,
        "category": pdf.parent.name,
        "pages": len(pages),
        "matched_terms": ", ".join(matched),
        "preview": clean[:1000]
    })


judgment_df = pd.DataFrame(
    judgment_summary
)

display(judgment_df)


# ==========================================================
# GOVERNMENT RESOLUTION / CIRCULAR IDENTIFICATION
# ==========================================================

print("\n" + "=" * 70)
print("GR / CIRCULAR DOCUMENTS")
print("=" * 70)

gr_summary = []

for pdf in gr_files:

    pages = get_pages(pdf)

    full_text = " ".join(
        text for _, text in pages
    )

    clean = " ".join(
        full_text.split()
    )

    gr_summary.append({
        "file": pdf.name,
        "pages": len(pages),
        "preview": clean[:1500]
    })


gr_df = pd.DataFrame(
    gr_summary
)

display(gr_df)

JUDGMENT DOCUMENTS


,file,category,pages,matched_terms,preview
0,Harish_Arora_WP_3433_2025.pdf,Cooperative,64,"Writ Petition, Petitioner, Respondent, High Co...",wp3433-2025-J Final.doc AGK IN THE HIGH COURT ...
1,Uday_Dalal_WP_15089_2025.pdf,Cooperative,30,"Writ Petition, Petitioner, Respondent, High Co...",wpl15089 & 15091-2025-J.doc AGK IN THE HIGH CO...
2,Best_Workers_Union_WP_1457_2015.pdf,Labour,23,"Writ Petition, Petitioner, Respondent, High Co...",wp1457-2015 ith wp9332-2015-J.doc AGK IN THE H...
3,Indo_Count_WP_4630_2025.pdf,Labour,18,"Writ Petition, Petitioner, Respondent, High Co...",k 1/18 909 wp 4630.25 n 4631.25 J as.doc IN TH...
4,Government_Land_WPL_33728_2022.pdf,Land Revenue,28,"Writ Petition, Petitioner, Respondent, High Co...",WPL.33728.22 HARSHADA SAWANT (P.A.) IN IN THE ...
5,Nirlon_Limited_WP_5398_2025.pdf,Land Revenue,21,"Writ Petition, Petitioner, Respondent, High Co...",Priyanka OSWP-5398-2025.doc IN THE HIGH COURT ...
6,Pravin_Gajanan_Thakur_WP_5716_2024.pdf,Land Revenue,36,"Writ Petition, Petitioner, Respondent, High Co...",Megha 903_wp_5716_2024_fc.docx IN THE HIGH COU...
7,Asian_Chemical_Industries_WP_11164_2024.pdf,MOFA,35,"Writ Petition, Petitioner, Respondent, High Co...",WP.11164.2024.doc Ajay IN IN THE THE HIGH HIGH...
8,Shivranjan_Towers_WP_11281_2025.pdf,MOFA,23,"Writ Petition, Petitioner, Respondent, High Co...",-WP-11281-2025.DOC Arun Sankpal IN THE HIGH CO...
9,Baburao_Gangadhar_Jadhav_MRTP.pdf,MRTP,9,"Writ Petition, Petitioner, Respondent, High Co...",SAGAR MAGAR WP-8829-2024.odt IN THE HIGH COURT...



GR / CIRCULAR DOCUMENTS


,file,pages,preview
0,Checklists for CriminalCivil Proposals.pdf,4,'_ ~. 'dlfrCt :(o9G ~~ :(o9~ -9G un K~A ~~~ Cp...
1,Manual for Handling Legislation and Subordinat...,398,M 32 i MANUAL FOR HANDLING LEGISLATION AND SUB...
2,Procedure in case reference made to the Rememb...,4,Procedure- In case of references Illude to the...
3,Rules of procedure for dealing of Legislation ...,60,Rules of procedure for dealing of Legislation ...


In [62]:
# Cell 35 — Targeted Search for Q26–Q30
# PyMuPDF only. No Gemini/API call.

import pymupdf
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)


def search_file(pdf_path, keywords):

    results = []

    doc = pymupdf.open(pdf_path)

    for page_no, page in enumerate(doc, start=1):

        text = page.get_text()
        lower = text.lower()

        matched = [
            k for k in keywords
            if k.lower() in lower
        ]

        if matched:

            results.append({
                "file": pdf_path.name,
                "category": pdf_path.parent.name,
                "page": page_no,
                "matched": ", ".join(matched),
                "text": " ".join(text.split())[:1000]
            })

    doc.close()

    return results


# ==========================================================
# Q26–Q28: CANDIDATE JUDGMENTS
# ==========================================================

print("=" * 70)
print("Q26–Q28 — JUDGMENT CANDIDATES")
print("=" * 70)

judgment_keywords = [
    "79A",
    "supersession",
    "managing committee",
    "committee",
    "co-operative",
    "eviction",
    "land revenue",
    "RERA",
    "slum",
    "MOFA",
    "developer",
    "tenant",
    "society"
]

judgment_results = []

for pdf in judgment_files:

    results = search_file(
        pdf,
        judgment_keywords
    )

    judgment_results.extend(results)


judgment_results_df = pd.DataFrame(
    judgment_results
)

if len(judgment_results_df) > 0:

    # Show strongest candidates first
    judgment_results_df["keyword_count"] = (
        judgment_results_df["matched"]
        .str.split(", ")
        .str.len()
    )

    judgment_results_df = (
        judgment_results_df
        .sort_values(
            "keyword_count",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(
        judgment_results_df.head(80)
    )

else:

    print("No matching judgment text found.")


# ==========================================================
# Q29–Q30: CANDIDATE GR / CIRCULARS
# ==========================================================

print("\n" + "=" * 70)
print("Q29–Q30 — GR / CIRCULAR CANDIDATES")
print("=" * 70)

gr_keywords = [
    "Government Resolution",
    "G.R.",
    "Resolution",
    "Circular",
    "direction",
    "policy",
    "issued",
    "Government of Maharashtra"
]

gr_results = []

for pdf in gr_files:

    results = search_file(
        pdf,
        gr_keywords
    )

    gr_results.extend(results)


gr_results_df = pd.DataFrame(
    gr_results
)

if len(gr_results_df) > 0:

    gr_results_df["keyword_count"] = (
        gr_results_df["matched"]
        .str.split(", ")
        .str.len()
    )

    gr_results_df = (
        gr_results_df
        .sort_values(
            "keyword_count",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(
        gr_results_df.head(80)
    )

else:

    print("No matching GR/Circular text found.")

Q26–Q28 — JUDGMENT CANDIDATES


,file,category,page,matched,text,keyword_count
0,Harish_Arora_WP_3433_2025.pdf,Cooperative,39,"79A, supersession, managing committee, committ...",wp3433-2025-J Final.doc Registrar to initiate ...,6
1,Harish_Arora_WP_3433_2025.pdf,Cooperative,5,"79A, managing committee, committee, developer,...",wp3433-2025-J Final.doc respondent No.10 on 20...,5
2,Harish_Arora_WP_3433_2025.pdf,Cooperative,30,"79A, managing committee, committee, developer,...",wp3433-2025-J Final.doc managing committee its...,5
3,Harish_Arora_WP_3433_2025.pdf,Cooperative,28,"79A, managing committee, committee, developer,...",wp3433-2025-J Final.doc statutory or instituti...,5
4,Harish_Arora_WP_3433_2025.pdf,Cooperative,31,"79A, managing committee, committee, developer,...",wp3433-2025-J Final.doc interpretation cannot ...,5
...,...,...,...,...,...,...
75,Asian_Chemical_Industries_WP_11164_2024.pdf,MOFA,16,"tenant, society",WP.11164.2024.doc denied Respondent No.1’s ent...,2
76,Asian_Chemical_Industries_WP_11164_2024.pdf,MOFA,18,"developer, society",WP.11164.2024.doc relied upon the following de...,2
77,Asian_Chemical_Industries_WP_11164_2024.pdf,MOFA,15,"MOFA, society",WP.11164.2024.doc 5.2. He would submit that Re...,2
78,Asian_Chemical_Industries_WP_11164_2024.pdf,MOFA,14,"tenant, society",WP.11164.2024.doc Petitioner surrendered all i...,2



Q29–Q30 — GR / CIRCULAR CANDIDATES


,file,category,page,matched,text,keyword_count
0,Rules of procedure for dealing of Legislation ...,Circulars and Government Resolutions issued by...,22,"Government Resolution, Resolution, Circular, p...",शासन ननर्णय क्रमाांकः एलईजी 2023/बी-594 पृष्ठ ...,5
1,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...,22,"Government Resolution, Resolution, policy, Gov...",M 32 xx legislative drafting. The contents of ...,4
2,Rules of procedure for dealing of Legislation ...,Circulars and Government Resolutions issued by...,1,"Government Resolution, Resolution, policy, Gov...",Rules of procedure for dealing of Legislation ...,4
3,Rules of procedure for dealing of Legislation ...,Circulars and Government Resolutions issued by...,4,"Resolution, Circular, issued, Government of Ma...",शासन ननर्णय क्रमाांकः एलईजी 2023/बी-594 पृष्ठ ...,4
4,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...,44,"Government Resolution, Resolution, issued",M 32 20 Publication of Act. 4.22. (1) When the...,3
...,...,...,...,...,...,...
75,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...,205,Government of Maharashtra,M 32 181 Delegation of powers Notification NOT...,1
76,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...,201,Government of Maharashtra,M 32 177 (8) Appointment of members Notificati...,1
77,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...,203,Government of Maharashtra,M 32 179 (9) Delegation of powers Order ORDER ...,1
78,Manual for Handling Legislation and Subordinat...,Circulars and Government Resolutions issued by...,198,Government of Maharashtra,M 32 174 (6) Essential Services Order ORDER In...,1


In [63]:
# Cell 36 — Identify Judgment & GR Titles / Case Details
# No Gemini/API call

import pymupdf
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")


# ==========================================================
# JUDGMENTS
# ==========================================================

judgment_files = sorted(
    (DATASET_DIR / "judgments" / "bombay_high_court").rglob("*.pdf")
)

judgment_info = []

for pdf in judgment_files:

    doc = pymupdf.open(pdf)

    # Read first 3 pages because case title/details
    # may not always be on page 1
    first_text = ""

    for i in range(min(3, len(doc))):
        first_text += "\n" + doc[i].get_text()

    doc.close()

    clean_text = " ".join(first_text.split())

    judgment_info.append({
        "file": pdf.name,
        "category": pdf.parent.name,
        "pages": len(pymupdf.open(pdf)),
        "first_1500_chars": clean_text[:1500]
    })


judgment_df = pd.DataFrame(judgment_info)

print("=" * 80)
print("ALL BOMBAY HIGH COURT JUDGMENTS")
print("=" * 80)

for i, row in judgment_df.iterrows():

    print(f"\n[{i}] {row['file']}")
    print(f"Category: {row['category']}")
    print(f"Pages: {row['pages']}")
    print(f"Text: {row['first_1500_chars'][:700]}")
    print("-" * 80)


# ==========================================================
# GR / CIRCULAR DOCUMENTS
# ==========================================================

gr_files = sorted(
    (
        DATASET_DIR /
        "Circulars and Government Resolutions issued by Department"
    ).rglob("*.pdf")
)

gr_info = []

for pdf in gr_files:

    doc = pymupdf.open(pdf)

    first_text = ""

    for i in range(min(5, len(doc))):
        first_text += "\n" + doc[i].get_text()

    doc.close()

    clean_text = " ".join(first_text.split())

    gr_info.append({
        "file": pdf.name,
        "category": pdf.parent.name,
        "pages": len(pymupdf.open(pdf)),
        "first_2000_chars": clean_text[:2000]
    })


gr_df = pd.DataFrame(gr_info)

print("\n" + "=" * 80)
print("ALL GR / CIRCULAR DOCUMENTS")
print("=" * 80)

for i, row in gr_df.iterrows():

    print(f"\n[{i}] {row['file']}")
    print(f"Pages: {row['pages']}")
    print(f"Text: {row['first_2000_chars'][:1200]}")
    print("-" * 80)

ALL BOMBAY HIGH COURT JUDGMENTS

[0] Harish_Arora_WP_3433_2025.pdf
Category: Cooperative
Pages: 64
Text: wp3433-2025-J Final.doc AGK IN THE HIGH COURT OF JUDICATURE AT BOMBAY CIVIL APPELLATE JURISDICTION WRIT PETITION NO.3433 OF 2025 1. Harish Arora, Age 45 years, Occupation: Business, residing at Flat Nos.1 & 2, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 2. Venetia Palia Age 50 years, Occupation: Business, residing at Flat No.3, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 3. Phiroze Karmali, Age 55 years, Occupation: Business, residing at Flat No.51, Trishul Building, 18-A, Kane Road, Bandra (West), Mumbai – 400 050 4. Khalid Omar Wakani, Age 65 years, Occupation: Business, residing at Darus-Salaam ‘A’, Plot No.197-B, Kane Road, Bandra (West), Mum
--------------------------------------------------------------------------------

[1] Uday_Dalal_WP_15089_2025.pdf
Category: Cooperative
Pages: 30
Text: wpl15089 & 15091-2025-J.doc AGK IN THE HI

In [64]:
# Cell 37 — Q26–Q30 Ground-Truth Candidate Mapping
# No Gemini/API call

import pymupdf
import pandas as pd
from pathlib import Path
import re

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

# ----------------------------------------------------------
# Helper
# ----------------------------------------------------------

def find_matches(pdf_path, keywords, max_results=10):

    doc = pymupdf.open(pdf_path)
    results = []

    for page_no, page in enumerate(doc, start=1):

        text = " ".join(page.get_text().split())
        lower = text.lower()

        matched = [
            k for k in keywords
            if k.lower() in lower
        ]

        if matched:
            results.append({
                "file": pdf_path.name,
                "page": page_no,
                "matched_keywords": ", ".join(matched),
                "keyword_count": len(matched),
                "text": text[:1200]
            })

    doc.close()

    return sorted(
        results,
        key=lambda x: x["keyword_count"],
        reverse=True
    )[:max_results]


# ==========================================================
# Q26 — ISSUE IN SELECTED BOMBAY HC JUDGMENT
# ==========================================================

print("=" * 80)
print("Q26 — ISSUE / SUBJECT OF JUDGMENT")
print("=" * 80)

q26_keywords = [
    "issue",
    "question",
    "challenge",
    "petition",
    "79A",
    "supersession",
    "managing committee",
    "co-operative society",
    "co-operative societies"
]

q26_results = []

for pdf in judgment_files:

    q26_results.extend(
        find_matches(
            pdf,
            q26_keywords,
            max_results=5
        )
    )

q26_df = pd.DataFrame(q26_results)

display(q26_df.head(20))


# ==========================================================
# Q27 — DECISION / ORDER OF SELECTED JUDGMENT
# ==========================================================

print("\n" + "=" * 80)
print("Q27 — DECISION / ORDER")
print("=" * 80)

q27_keywords = [
    "ORDER",
    "disposed of",
    "allowed",
    "dismissed",
    "petition is allowed",
    "petition is dismissed",
    "rule is made absolute",
    "rule is discharged",
    "impugned order",
    "directions"
]

q27_results = []

for pdf in judgment_files:

    q27_results.extend(
        find_matches(
            pdf,
            q27_keywords,
            max_results=5
        )
    )

q27_df = pd.DataFrame(q27_results)

display(q27_df.head(20))


# ==========================================================
# Q28 — LEGAL PROVISION IN SELECTED JUDGMENT
# ==========================================================

print("\n" + "=" * 80)
print("Q28 — LEGAL PROVISION")
print("=" * 80)

q28_keywords = [
    "section 79A",
    "section 79",
    "Maharashtra Co-operative Societies Act",
    "Maharashtra Cooperative Societies Act",
    "Maharashtra Rent Control Act",
    "MOFA",
    "RERA",
    "MRTP Act",
    "Bombay High Court"
]

q28_results = []

for pdf in judgment_files:

    q28_results.extend(
        find_matches(
            pdf,
            q28_keywords,
            max_results=5
        )
    )

q28_df = pd.DataFrame(q28_results)

display(q28_df.head(20))


# ==========================================================
# Q29 — SUBJECT OF SELECTED GR
# ==========================================================

print("\n" + "=" * 80)
print("Q29 — GR SUBJECT")
print("=" * 80)

q29_keywords = [
    "Government Resolution",
    "G.R.",
    "subject",
    "regarding",
    "policy",
    "procedure",
    "legislation",
    "subordinate legislation",
    "criminal",
    "civil",
    "proposals"
]

q29_results = []

for pdf in gr_files:

    q29_results.extend(
        find_matches(
            pdf,
            q29_keywords,
            max_results=8
        )
    )

q29_df = pd.DataFrame(q29_results)

display(q29_df.head(25))


# ==========================================================
# Q30 — DIRECTION / POLICY IN SELECTED GR
# ==========================================================

print("\n" + "=" * 80)
print("Q30 — GR DIRECTION / POLICY")
print("=" * 80)

q30_keywords = [
    "direction",
    "directions",
    "policy",
    "procedure",
    "shall",
    "Government Resolution",
    "implementation",
    "compliance",
    "instructions",
    "guidelines"
]

q30_results = []

for pdf in gr_files:

    q30_results.extend(
        find_matches(
            pdf,
            q30_keywords,
            max_results=8
        )
    )

q30_df = pd.DataFrame(q30_results)

display(q30_df.head(25))

Q26 — ISSUE / SUBJECT OF JUDGMENT


,file,page,matched_keywords,keyword_count,text
0,Harish_Arora_WP_3433_2025.pdf,21,"issue, question, challenge, petition, 79A, man...",6,wp3433-2025-J Final.doc the whole exercise fut...
1,Harish_Arora_WP_3433_2025.pdf,4,"question, challenge, petition, 79A, managing c...",5,wp3433-2025-J Final.doc Mr. Mayur Khandeparkar...
2,Harish_Arora_WP_3433_2025.pdf,5,"issue, question, petition, 79A, managing commi...",5,wp3433-2025-J Final.doc respondent No.10 on 20...
3,Harish_Arora_WP_3433_2025.pdf,15,"question, petition, 79A, supersession, managin...",5,wp3433-2025-J Final.doc petition and for the p...
4,Harish_Arora_WP_3433_2025.pdf,39,"issue, petition, 79A, supersession, managing c...",5,wp3433-2025-J Final.doc Registrar to initiate ...
5,Uday_Dalal_WP_15089_2025.pdf,5,"issue, question, challenge, petition, managing...",5,wpl15089 & 15091-2025-J.doc East corner of the...
6,Uday_Dalal_WP_15089_2025.pdf,4,"issue, challenge, petition",3,wpl15089 & 15091-2025-J.doc Mr. A.A. Alaspurka...
7,Uday_Dalal_WP_15089_2025.pdf,6,"issue, petition, managing committee",3,wpl15089 & 15091-2025-J.doc 21 July 2025 and s...
8,Uday_Dalal_WP_15089_2025.pdf,7,"challenge, petition, managing committee",3,wpl15089 & 15091-2025-J.doc 30 September 2025....
9,Uday_Dalal_WP_15089_2025.pdf,11,"question, challenge, petition",3,"wpl15089 & 15091-2025-J.doc in Daman Singh, he..."



Q27 — DECISION / ORDER


,file,page,matched_keywords,keyword_count,text
0,Harish_Arora_WP_3433_2025.pdf,14,"ORDER, dismissed, impugned order",3,wp3433-2025-J Final.doc that the Chairman of t...
1,Harish_Arora_WP_3433_2025.pdf,43,"ORDER, allowed, impugned order",3,wp3433-2025-J Final.doc is not arbitrary but g...
2,Harish_Arora_WP_3433_2025.pdf,62,"ORDER, impugned order, directions",3,wp3433-2025-J Final.doc 103. Having regard to ...
3,Harish_Arora_WP_3433_2025.pdf,9,"ORDER, impugned order",2,wp3433-2025-J Final.doc further prejudice to t...
4,Harish_Arora_WP_3433_2025.pdf,10,"ORDER, impugned order",2,wp3433-2025-J Final.doc made in paragraph 19 t...
5,Uday_Dalal_WP_15089_2025.pdf,29,"ORDER, allowed, directions",3,"wpl15089 & 15091-2025-J.doc transferable, give..."
6,Uday_Dalal_WP_15089_2025.pdf,6,"ORDER, allowed",2,wpl15089 & 15091-2025-J.doc 21 July 2025 and s...
7,Uday_Dalal_WP_15089_2025.pdf,5,ORDER,1,wpl15089 & 15091-2025-J.doc East corner of the...
8,Uday_Dalal_WP_15089_2025.pdf,7,ORDER,1,wpl15089 & 15091-2025-J.doc 30 September 2025....
9,Uday_Dalal_WP_15089_2025.pdf,8,ORDER,1,wpl15089 & 15091-2025-J.doc membership applica...



Q28 — LEGAL PROVISION


,file,page,matched_keywords,keyword_count,text
0,Harish_Arora_WP_3433_2025.pdf,4,"section 79A, section 79, Maharashtra Cooperati...",3,wp3433-2025-J Final.doc Mr. Mayur Khandeparkar...
1,Harish_Arora_WP_3433_2025.pdf,28,"section 79A, section 79, Maharashtra Cooperati...",3,wp3433-2025-J Final.doc statutory or instituti...
2,Harish_Arora_WP_3433_2025.pdf,40,"section 79A, section 79, Maharashtra Cooperati...",3,wp3433-2025-J Final.doc clearly misconceived a...
3,Harish_Arora_WP_3433_2025.pdf,41,"section 79A, section 79, Maharashtra Cooperati...",3,wp3433-2025-J Final.doc statute does not expre...
4,Harish_Arora_WP_3433_2025.pdf,43,"section 79A, section 79, Maharashtra Cooperati...",3,wp3433-2025-J Final.doc is not arbitrary but g...
5,Uday_Dalal_WP_15089_2025.pdf,4,Maharashtra Cooperative Societies Act,1,wpl15089 & 15091-2025-J.doc Mr. A.A. Alaspurka...
6,Uday_Dalal_WP_15089_2025.pdf,28,MOFA,1,wpl15089 & 15091-2025-J.doc body can cure the ...
7,Nirlon_Limited_WP_5398_2025.pdf,2,Bombay High Court,1,Priyanka OSWP-5398-2025.doc 4. Additional Comm...
8,Nirlon_Limited_WP_5398_2025.pdf,18,Bombay High Court,1,Priyanka OSWP-5398-2025.doc Court to discourag...
9,Asian_Chemical_Industries_WP_11164_2024.pdf,1,MOFA,1,WP.11164.2024.doc Ajay IN IN THE THE HIGH HIGH...



Q29 — GR SUBJECT


,file,page,matched_keywords,keyword_count,text
0,Checklists for CriminalCivil Proposals.pdf,2,civil,1,A CIVIL J I I Points to consider (Check list) ...
1,Checklists for CriminalCivil Proposals.pdf,3,criminal,1,A I I Points to consider 1 check list for crim...
2,Checklists for CriminalCivil Proposals.pdf,4,regarding,1,", I 13 Copy of deposition of all witnesses, 14..."
3,Manual for Handling Legislation and Subordinat...,21,"subject, procedure, legislation, subordinate l...",5,M 32 xix INTRODUCTION The Government Departmen...
4,Manual for Handling Legislation and Subordinat...,22,"Government Resolution, policy, procedure, legi...",5,M 32 xx legislative drafting. The contents of ...
5,Manual for Handling Legislation and Subordinat...,26,"regarding, policy, legislation, subordinate le...",5,M 32 2 (15) “General Clauses Act” means the Ma...
6,Manual for Handling Legislation and Subordinat...,152,"subject, policy, procedure, legislation, subor...",5,M 32 128 (d) Rate-making function provided ade...
7,Manual for Handling Legislation and Subordinat...,3,"subject, procedure, legislation, subordinate l...",4,M 32 iii PREFACE Law is a set of rules that ar...
8,Manual for Handling Legislation and Subordinat...,5,"subject, procedure, legislation, subordinate l...",4,"M 32 v DECLARATION I hereby declare that, this..."
9,Manual for Handling Legislation and Subordinat...,11,"procedure, legislation, subordinate legislatio...",4,M 32 MANUAL FOR HANDALING LEGISLATION AND SUBO...



Q30 — GR DIRECTION / POLICY


,file,page,matched_keywords,keyword_count,text
0,Manual for Handling Legislation and Subordinat...,22,"policy, procedure, Government Resolution, impl...",4,M 32 xx legislative drafting. The contents of ...
1,Manual for Handling Legislation and Subordinat...,134,"policy, procedure, shall, compliance",4,M 32 110 Procedure of making reference relatin...
2,Manual for Handling Legislation and Subordinat...,5,"procedure, instructions, guidelines",3,"M 32 v DECLARATION I hereby declare that, this..."
3,Manual for Handling Legislation and Subordinat...,14,"procedure, Government Resolution, guidelines",3,M 32 xii CHAPTER XVII 17 Guidelines for drafti...
4,Manual for Handling Legislation and Subordinat...,25,"direction, directions, instructions",3,M 32 1 CHAPTER I DEFINITIONS 1. Definitions.––...
5,Manual for Handling Legislation and Subordinat...,33,"policy, procedure, shall",3,M 32 9 CHAPTER IV BILL PROCESSING STAGES Artic...
6,Manual for Handling Legislation and Subordinat...,45,"procedure, shall, implementation",3,M 32 21 Making of Rules. 4.26. (1) If the Act ...
7,Manual for Handling Legislation and Subordinat...,53,"policy, procedure, shall",3,M 32 29 CHAPTER V ORDINANCE PROCESSING STAGES ...
8,Procedure in case reference made to the Rememb...,2,"policy, procedure, compliance",3,~I 'I Rule 8: Matters on which opinion of Reme...
9,Procedure in case reference made to the Rememb...,1,"procedure, instructions",2,Procedure- In case of references Illude to the...


In [67]:
# ==============================================================
# FINAL CORRECTED RETRIEVAL EVALUATION
# Precision@5 + Recall@5
# No Gemini/API call
# ==============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------------------
# PATHS
# --------------------------------------------------------------

EVAL_DIR = Path("/content/drive/MyDrive/Nyay_AI/evaluation")

RETRIEVAL_FILE = (
    EVAL_DIR / "official_30_retrieval_results.csv"
)

OUTPUT_FILE = (
    EVAL_DIR / "final_correct_retrieval_metrics.csv"
)


# --------------------------------------------------------------
# GROUND-TRUTH DOCUMENTS
# --------------------------------------------------------------

ground_truth_docs = {

    1: "DOC_020",
    2: "DOC_020",
    3: "DOC_020",
    4: "DOC_020",
    5: "DOC_020",
    6: "DOC_020",
    7: "DOC_020",

    8: "DOC_013",
    9: "DOC_013",
    10: "DOC_013",
    11: "DOC_013",
    12: "DOC_013",

    13: "DOC_024",
    14: "DOC_024",
    15: "DOC_024",
    16: "DOC_024",
    17: "DOC_024",

    18: "DOC_021",
    19: "DOC_021",
    20: "DOC_021",

    21: "DOC_019",
    22: "DOC_019",
    23: "DOC_019",

    24: "DOC_028",
    25: "DOC_028",

    26: "DOC_053",
    27: "DOC_053",
    28: "DOC_053",

    29: "DOC_002",
    30: "DOC_002"
}


# --------------------------------------------------------------
# LOAD RETRIEVAL RESULTS
# --------------------------------------------------------------

print("=" * 80)
print("CORRECTED PRECISION@5 / RECALL@5 EVALUATION")
print("=" * 80)

retrieval_df = pd.read_csv(RETRIEVAL_FILE)

print(
    f"Loaded rows: {len(retrieval_df)}"
)

print(
    "\nColumns:"
)

print(
    retrieval_df.columns.tolist()
)


# --------------------------------------------------------------
# HELPER — SPLIT TOP-5 DOCUMENT IDS
# --------------------------------------------------------------

def parse_doc_ids(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    # Handle the format:
    # DOC_020 | DOC_064 | DOC_020 | DOC_020 | DOC_015

    parts = value.replace(
        ";", "|"
    ).split("|")

    docs = []

    for p in parts:

        p = p.strip()

        if p:
            docs.append(p)

    return docs[:5]


# --------------------------------------------------------------
# FIND COLUMNS
# --------------------------------------------------------------

question_col = "question_id"
doc_col = "retrieved_document_ids"

if question_col not in retrieval_df.columns:
    raise ValueError(
        "question_id column not found."
    )

if doc_col not in retrieval_df.columns:
    raise ValueError(
        "retrieved_document_ids column not found."
    )


# --------------------------------------------------------------
# CALCULATE METRICS
# --------------------------------------------------------------

results = []

for _, row in retrieval_df.iterrows():

    qid = int(row[question_col])

    retrieved_docs = parse_doc_ids(
        row[doc_col]
    )

    # Ensure exactly top-5 conceptually
    top5 = retrieved_docs[:5]

    gt_doc = ground_truth_docs.get(qid)

    # Count retrieved results matching the
    # ground-truth document.
    #
    # IMPORTANT:
    # Repeated chunks from the same document
    # are counted as retrieved relevant chunks
    # for Precision@5.

    relevant_hits = sum(
        1
        for doc in top5
        if doc == gt_doc
    )

    precision_at_5 = (
        relevant_hits / 5
        if len(top5) == 5
        else (
            relevant_hits / len(top5)
            if len(top5) > 0
            else 0
        )
    )

    # Document-level Recall@5:
    # The question has one ground-truth document.
    #
    # If that document appears anywhere in top-5,
    # recall = 1.0.

    recall_at_5 = (
        1.0
        if gt_doc in top5
        else 0.0
    )

    results.append({

        "question_id": qid,

        "ground_truth_document": gt_doc,

        "retrieved_top_5": " | ".join(top5),

        "relevant_hits_in_top5": relevant_hits,

        "precision_at_5": precision_at_5,

        "recall_at_5": recall_at_5
    })


metrics_df = pd.DataFrame(results)


# --------------------------------------------------------------
# ADD SIMILARITY + LATENCY
# --------------------------------------------------------------

if "max_similarity" in retrieval_df.columns:

    similarity_map = (
        retrieval_df
        .set_index(question_col)["max_similarity"]
    )

    metrics_df["max_similarity"] = (
        metrics_df["question_id"]
        .map(similarity_map)
    )

if "latency_seconds" in retrieval_df.columns:

    latency_map = (
        retrieval_df
        .set_index(question_col)["latency_seconds"]
    )

    metrics_df["latency_seconds"] = (
        metrics_df["question_id"]
        .map(latency_map)
    )


# --------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------

mean_precision = metrics_df[
    "precision_at_5"
].mean()

mean_recall = metrics_df[
    "recall_at_5"
].mean()

mean_similarity = metrics_df[
    "max_similarity"
].mean()

mean_latency = metrics_df[
    "latency_seconds"
].mean()


retrieval_success = (
    metrics_df["retrieved_top_5"]
    .str.len()
    .gt(0)
    .sum()
)


questions_with_gt_in_top5 = (
    metrics_df["recall_at_5"] == 1.0
).sum()


# --------------------------------------------------------------
# PRINT FINAL RESULTS
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL RETRIEVAL RESULTS")
print("=" * 80)

print(
    f"Questions evaluated       : {len(metrics_df)}"
)

print(
    f"Successful retrievals     : "
    f"{retrieval_success}/{len(metrics_df)}"
)

print(
    f"Ground-truth in Top-5     : "
    f"{questions_with_gt_in_top5}/{len(metrics_df)}"
)

print(
    f"Mean Precision@5          : "
    f"{mean_precision:.4f}"
)

print(
    f"Mean Recall@5             : "
    f"{mean_recall:.4f}"
)

print(
    f"Mean Max Similarity       : "
    f"{mean_similarity:.4f}"
)

print(
    f"Mean Retrieval Latency    : "
    f"{mean_latency:.4f} sec"
)


# --------------------------------------------------------------
# PERCENTAGES
# --------------------------------------------------------------

print("\n" + "-" * 80)
print("PAPER-READY PERCENTAGES")
print("-" * 80)

print(
    f"Precision@5              : "
    f"{mean_precision * 100:.2f}%"
)

print(
    f"Recall@5                 : "
    f"{mean_recall * 100:.2f}%"
)

print(
    f"Top-5 Retrieval Coverage : "
    f"{questions_with_gt_in_top5 / len(metrics_df) * 100:.2f}%"
)


# --------------------------------------------------------------
# SHOW QUESTION-BY-QUESTION RESULTS
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("QUESTION-BY-QUESTION RESULTS")
print("=" * 80)

display(metrics_df)


# --------------------------------------------------------------
# SAVE
# --------------------------------------------------------------

metrics_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "\nSaved corrected results to:"
)

print(
    OUTPUT_FILE
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

CORRECTED PRECISION@5 / RECALL@5 EVALUATION
Loaded rows: 30

Columns:
['question_id', 'question', 'retrieved_document_ids', 'retrieved_pages', 'relevant_document_ids', 'precision_at_5', 'recall_at_5', 'max_similarity', 'latency_seconds', 'status']

FINAL RETRIEVAL RESULTS
Questions evaluated       : 30
Successful retrievals     : 30/30
Ground-truth in Top-5     : 30/30
Mean Precision@5          : 0.7667
Mean Recall@5             : 1.0000
Mean Max Similarity       : 0.7364
Mean Retrieval Latency    : 0.7361 sec

--------------------------------------------------------------------------------
PAPER-READY PERCENTAGES
--------------------------------------------------------------------------------
Precision@5              : 76.67%
Recall@5                 : 100.00%
Top-5 Retrieval Coverage : 100.00%

QUESTION-BY-QUESTION RESULTS


,question_id,ground_truth_document,retrieved_top_5,relevant_hits_in_top5,precision_at_5,recall_at_5,max_similarity,latency_seconds
0,1,DOC_020,DOC_020 | DOC_020 | DOC_020 | DOC_020 | DOC_020,5,1.0,1.0,0.8963,1.0805
1,2,DOC_020,DOC_020 | DOC_064 | DOC_020 | DOC_020 | DOC_015,3,0.6,1.0,0.8052,0.7813
2,3,DOC_020,DOC_020 | DOC_020 | DOC_020 | DOC_002 | DOC_015,3,0.6,1.0,0.8194,0.5920
3,4,DOC_020,DOC_020 | DOC_020 | DOC_020 | DOC_014 | DOC_020,4,0.8,1.0,0.8094,0.4532
4,5,DOC_020,DOC_020 | DOC_002 | DOC_020 | DOC_020 | DOC_020,4,0.8,1.0,0.7900,0.5665
5,6,DOC_020,DOC_020 | DOC_020 | DOC_020 | DOC_020 | DOC_020,5,1.0,1.0,0.7887,0.5928
6,7,DOC_020,DOC_020 | DOC_002 | DOC_020 | DOC_020 | DOC_020,4,0.8,1.0,0.8110,0.5662
7,8,DOC_013,DOC_013 | DOC_013 | DOC_013 | DOC_013 | DOC_013,5,1.0,1.0,0.8241,0.5394
8,9,DOC_013,DOC_013 | DOC_013 | DOC_013 | DOC_013 | DOC_013,5,1.0,1.0,0.7127,0.6557
9,10,DOC_013,DOC_013 | DOC_013 | DOC_013 | DOC_010 | DOC_013,4,0.8,1.0,0.7816,0.5920



Saved corrected results to:
/content/drive/MyDrive/Nyay_AI/evaluation/final_correct_retrieval_metrics.csv

DONE


In [69]:
# ==============================================================
# FINAL CORRECTED EVALUATION — USE GEMINI RESULTS ONLY
# No API call
# ==============================================================

import pandas as pd
import numpy as np
from pathlib import Path

EVAL_DIR = Path("/content/drive/MyDrive/Nyay_AI/evaluation")

GEMINI_FILE = EVAL_DIR / "official_30_gemini_answers.csv"
RETRIEVAL_FILE = EVAL_DIR / "final_correct_retrieval_metrics.csv"

print("=" * 80)
print("NYAY AI — FINAL CORRECTED EVALUATION")
print("=" * 80)


# ==============================================================
# 1. LOAD RETRIEVAL RESULTS
# ==============================================================

retrieval_df = pd.read_csv(RETRIEVAL_FILE)

print("\nRetrieval:")
print(f"Questions: {len(retrieval_df)}")


# ==============================================================
# 2. LOAD GEMINI RESULTS
# ==============================================================

if not GEMINI_FILE.exists():
    raise FileNotFoundError(
        f"Gemini evaluation file not found:\n{GEMINI_FILE}"
    )

gemini_df = pd.read_csv(GEMINI_FILE)

print("\nGemini answer evaluation:")
print(f"Rows: {len(gemini_df)}")

print("\nGemini columns:")
print(gemini_df.columns.tolist())


# ==============================================================
# 3. REMOVE API-ERROR ROWS IF ANY
# ==============================================================

if "status" in gemini_df.columns:

    print("\nStatus distribution:")
    print(
        gemini_df["status"].value_counts(
            dropna=False
        )
    )

    valid_gemini = gemini_df[
        ~gemini_df["status"]
        .astype(str)
        .str.contains(
            "error",
            case=False,
            na=False
        )
    ].copy()

else:

    valid_gemini = gemini_df.copy()


print(
    f"\nValid Gemini rows: {len(valid_gemini)}"
)


# ==============================================================
# 4. NUMERIC CONVERSION
# ==============================================================

numeric_columns = [
    "citation_format_accuracy",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "max_similarity"
]

for col in numeric_columns:

    if col in valid_gemini.columns:

        valid_gemini[col] = pd.to_numeric(
            valid_gemini[col],
            errors="coerce"
        )


# ==============================================================
# 5. CALCULATE TOTAL TOKENS
# ==============================================================

if (
    "input_tokens" in valid_gemini.columns
    and
    "output_tokens" in valid_gemini.columns
):

    valid_gemini["total_tokens"] = (
        valid_gemini["input_tokens"].fillna(0)
        +
        valid_gemini["output_tokens"].fillna(0)
    )


# ==============================================================
# 6. ANSWER METRICS
# ==============================================================

print("\n" + "=" * 80)
print("ANSWER GENERATION METRICS")
print("=" * 80)


if "citation_format_accuracy" in valid_gemini.columns:

    citation_accuracy = (
        valid_gemini[
            "citation_format_accuracy"
        ].mean()
    )

    print(
        f"Citation Accuracy       : "
        f"{citation_accuracy:.4f} "
        f"({citation_accuracy * 100:.2f}%)"
    )

else:

    citation_accuracy = np.nan

    print(
        "Citation Accuracy       : Not available"
    )


if "input_tokens" in valid_gemini.columns:

    mean_input_tokens = (
        valid_gemini["input_tokens"].mean()
    )

    print(
        f"Mean Input Tokens       : "
        f"{mean_input_tokens:.2f}"
    )

else:

    mean_input_tokens = np.nan


if "output_tokens" in valid_gemini.columns:

    mean_output_tokens = (
        valid_gemini["output_tokens"].mean()
    )

    print(
        f"Mean Output Tokens      : "
        f"{mean_output_tokens:.2f}"
    )

else:

    mean_output_tokens = np.nan


if "total_tokens" in valid_gemini.columns:

    mean_total_tokens = (
        valid_gemini["total_tokens"].mean()
    )

    print(
        f"Mean Total Tokens       : "
        f"{mean_total_tokens:.2f}"
    )

else:

    mean_total_tokens = np.nan


if "latency_seconds" in valid_gemini.columns:

    mean_answer_latency = (
        valid_gemini["latency_seconds"].mean()
    )

    print(
        f"Mean Answer Latency     : "
        f"{mean_answer_latency:.4f} sec"
    )

else:

    mean_answer_latency = np.nan


# ==============================================================
# 7. CONFIDENCE
# ==============================================================

if "confidence" in valid_gemini.columns:

    print("\n" + "-" * 80)
    print("CONFIDENCE DISTRIBUTION")
    print("-" * 80)

    confidence_counts = (
        valid_gemini[
            "confidence"
        ]
        .value_counts(
            dropna=False
        )
    )

    print(
        confidence_counts
    )

else:

    confidence_counts = None


# ==============================================================
# 8. MERGE RETRIEVAL + GEMINI
# ==============================================================

print("\n" + "=" * 80)
print("MERGING RETRIEVAL + GEMINI RESULTS")
print("=" * 80)

final_df = retrieval_df.merge(
    valid_gemini,
    on="question_id",
    how="left",
    suffixes=(
        "_retrieval",
        "_gemini"
    )
)

print(
    f"Final rows: {len(final_df)}"
)


# ==============================================================
# 9. CHECK MISSING GEMINI ANSWERS
# ==============================================================

missing_answers = 0

if "answer" in final_df.columns:

    missing_answers = (
        final_df["answer"]
        .isna()
        .sum()
    )

print(
    f"Missing Gemini answers: "
    f"{missing_answers}"
)


# ==============================================================
# 10. FINAL RETRIEVAL METRICS
# ==============================================================

precision = (
    retrieval_df["precision_at_5"]
    .mean()
)

recall = (
    retrieval_df["recall_at_5"]
    .mean()
)

similarity = (
    retrieval_df["max_similarity"]
    .mean()
)

retrieval_latency = (
    retrieval_df["latency_seconds"]
    .mean()
)


# ==============================================================
# 11. FINAL PAPER-READY SUMMARY
# ==============================================================

print("\n" + "=" * 80)
print("NYAY AI — FINAL PAPER-READY RESULTS")
print("=" * 80)

print(
    f"\nQuestions Evaluated       : "
    f"{len(retrieval_df)}"
)

print(
    f"Retrieval Success         : "
    f"{len(retrieval_df)}/{len(retrieval_df)}"
)

print(
    f"Precision@5               : "
    f"{precision * 100:.2f}%"
)

print(
    f"Recall@5                  : "
    f"{recall * 100:.2f}%"
)

print(
    f"Top-5 Retrieval Coverage  : "
    f"{recall * 100:.2f}%"
)

print(
    f"Mean Similarity           : "
    f"{similarity:.4f}"
)

print(
    f"Mean Retrieval Latency    : "
    f"{retrieval_latency:.4f} sec"
)

if not pd.isna(citation_accuracy):

    print(
        f"Citation Accuracy        : "
        f"{citation_accuracy * 100:.2f}%"
    )

if not pd.isna(mean_input_tokens):

    print(
        f"Mean Input Tokens        : "
        f"{mean_input_tokens:.2f}"
    )

if not pd.isna(mean_output_tokens):

    print(
        f"Mean Output Tokens       : "
        f"{mean_output_tokens:.2f}"
    )

if not pd.isna(mean_total_tokens):

    print(
        f"Mean Total Tokens        : "
        f"{mean_total_tokens:.2f}"
    )

if not pd.isna(mean_answer_latency):

    print(
        f"Mean Answer Latency      : "
        f"{mean_answer_latency:.4f} sec"
    )


# ==============================================================
# 12. SAVE CORRECT FINAL FILES
# ==============================================================

correct_answer_file = (
    EVAL_DIR /
    "final_30_gemini_answer_evaluation.csv"
)

correct_complete_file = (
    EVAL_DIR /
    "Nyay_AI_FINAL_COMPLETE_EVALUATION_CORRECT.csv"
)

summary_file = (
    EVAL_DIR /
    "Nyay_AI_FINAL_PAPER_METRICS.txt"
)


valid_gemini.to_csv(
    correct_answer_file,
    index=False
)

final_df.to_csv(
    correct_complete_file,
    index=False
)


# ==============================================================
# 13. SAVE TEXT SUMMARY
# ==============================================================

with open(
    summary_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "NYAY AI — FINAL PAPER METRICS\n"
    )

    f.write(
        "=" * 60 + "\n\n"
    )

    f.write(
        f"Questions evaluated: "
        f"{len(retrieval_df)}\n"
    )

    f.write(
        f"Retrieval success: "
        f"{len(retrieval_df)}/{len(retrieval_df)}\n"
    )

    f.write(
        f"Precision@5: "
        f"{precision * 100:.2f}%\n"
    )

    f.write(
        f"Recall@5: "
        f"{recall * 100:.2f}%\n"
    )

    f.write(
        f"Mean similarity: "
        f"{similarity:.4f}\n"
    )

    f.write(
        f"Mean retrieval latency: "
        f"{retrieval_latency:.4f} sec\n"
    )

    if not pd.isna(citation_accuracy):

        f.write(
            f"Citation accuracy: "
            f"{citation_accuracy * 100:.2f}%\n"
        )

    if not pd.isna(mean_input_tokens):

        f.write(
            f"Mean input tokens: "
            f"{mean_input_tokens:.2f}\n"
        )

    if not pd.isna(mean_output_tokens):

        f.write(
            f"Mean output tokens: "
            f"{mean_output_tokens:.2f}\n"
        )

    if not pd.isna(mean_total_tokens):

        f.write(
            f"Mean total tokens: "
            f"{mean_total_tokens:.2f}\n"
        )

    if not pd.isna(mean_answer_latency):

        f.write(
            f"Mean answer latency: "
            f"{mean_answer_latency:.4f} sec\n"
        )


# ==============================================================
# 14. FILES
# ==============================================================

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    correct_answer_file
)

print(
    correct_complete_file
)

print(
    summary_file
)

print("\n" + "=" * 80)
print("COMPLETE")
print("=" * 80)

print(
    "\nIMPORTANT: Gemini was NOT called again."
)

NYAY AI — FINAL CORRECTED EVALUATION

Retrieval:
Questions: 30

Gemini answer evaluation:
Rows: 30

Gemini columns:
['question_id', 'question', 'answer', 'confidence', 'max_similarity', 'citation_format_accuracy', 'input_tokens', 'output_tokens', 'total_tokens', 'latency_seconds', 'status', 'error']

Status distribution:
status
Success    30
Name: count, dtype: int64

Valid Gemini rows: 30

ANSWER GENERATION METRICS
Citation Accuracy       : 0.8333 (83.33%)
Mean Input Tokens       : 2259.87
Mean Output Tokens      : 184.73
Mean Total Tokens       : 2444.60
Mean Answer Latency     : 34.2237 sec

--------------------------------------------------------------------------------
CONFIDENCE DISTRIBUTION
--------------------------------------------------------------------------------
confidence
Corpus-Grounded          20
Insufficient Evidence    10
Name: count, dtype: int64

MERGING RETRIEVAL + GEMINI RESULTS
Final rows: 30
Missing Gemini answers: 0

NYAY AI — FINAL PAPER-READY RESULTS

Ques

In [70]:
# ==============================================================
# FINAL CELL — CREATE COMPLETE NYAY AI UI / PROJECT ZIP
# ==============================================================

from google.colab import drive
from pathlib import Path
import shutil
import os
import json
import zipfile
from datetime import datetime

# --------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# --------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

# --------------------------------------------------------------
# 2. PROJECT PATHS
# --------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/Nyay_AI")

EVAL_DIR = PROJECT / "evaluation"
MAIN_DB = PROJECT / "chroma_final"
USER_DB = PROJECT / "user_upload_chroma"

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

PACKAGE_DIR = PROJECT / "final_package"
ZIP_FILE = PROJECT / "Nyay_AI_FINAL_UI_PACKAGE.zip"

# Clean old package folder
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 3. CREATE PACKAGE STRUCTURE
# --------------------------------------------------------------

folders = [
    "notebook",
    "evaluation",
    "database",
    "dataset_info",
    "backend",
    "frontend",
    "config",
    "documentation"
]

for folder in folders:
    (PACKAGE_DIR / folder).mkdir(
        parents=True,
        exist_ok=True
    )

# --------------------------------------------------------------
# 4. COPY FINAL NOTEBOOK(S)
# --------------------------------------------------------------

print("=" * 80)
print("PACKAGING NYAY AI")
print("=" * 80)

notebook_candidates = [
    Path("/content/Nyay_AI_Groq_FINAL_Complete (1)(1).ipynb"),
    Path("/content/Nyay_AI_FINAL.ipynb")
]

copied_notebooks = []

for notebook in notebook_candidates:

    if notebook.exists():

        destination = (
            PACKAGE_DIR /
            "notebook" /
            notebook.name
        )

        shutil.copy2(
            notebook,
            destination
        )

        copied_notebooks.append(
            destination.name
        )

# Also search /content for notebooks
for notebook in Path("/content").glob("*.ipynb"):

    destination = (
        PACKAGE_DIR /
        "notebook" /
        notebook.name
    )

    if not destination.exists():

        shutil.copy2(
            notebook,
            destination
        )

        copied_notebooks.append(
            notebook.name
        )

print(
    f"\nNotebooks copied: {len(copied_notebooks)}"
)

# --------------------------------------------------------------
# 5. COPY FINAL EVALUATION FILES
# --------------------------------------------------------------

evaluation_files = [
    "evaluation_questions.csv",
    "official_30_retrieval_results.csv",
    "official_30_gemini_answers.csv",
    "final_correct_retrieval_metrics.csv",
    "final_30_gemini_answer_evaluation.csv",
    "Nyay_AI_FINAL_COMPLETE_EVALUATION_CORRECT.csv",
    "Nyay_AI_FINAL_PAPER_METRICS.txt",
    "Nyay_AI_FINAL_EVALUATION_SUMMARY.txt"
]

copied_evaluation = []

for filename in evaluation_files:

    source = EVAL_DIR / filename

    if source.exists():

        destination = (
            PACKAGE_DIR /
            "evaluation" /
            filename
        )

        shutil.copy2(
            source,
            destination
        )

        copied_evaluation.append(filename)

print(
    f"Evaluation files copied: "
    f"{len(copied_evaluation)}"
)

# --------------------------------------------------------------
# 6. COPY CHROMADB
# --------------------------------------------------------------

def copy_directory_if_exists(source, destination):

    if source.exists():

        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True
        )

        return True

    return False


main_db_copied = copy_directory_if_exists(
    MAIN_DB,
    PACKAGE_DIR / "database" / "chroma_final"
)

user_db_copied = copy_directory_if_exists(
    USER_DB,
    PACKAGE_DIR / "database" / "user_upload_chroma"
)

print(
    f"Main ChromaDB copied: {main_db_copied}"
)

print(
    f"User-upload ChromaDB copied: {user_db_copied}"
)

# --------------------------------------------------------------
# 7. DATASET INFORMATION
# --------------------------------------------------------------
# We do NOT copy the complete 65/68 PDF dataset into the ZIP.
# This keeps the UI package manageable.
#
# Instead we create a complete inventory.

dataset_inventory = []

if DATASET_DIR.exists():

    for pdf in sorted(
        DATASET_DIR.rglob("*.pdf")
    ):

        try:

            relative_path = str(
                pdf.relative_to(DATASET_DIR)
            )

        except Exception:

            relative_path = pdf.name

        dataset_inventory.append({

            "filename": pdf.name,

            "relative_path":
                relative_path,

            "size_MB":
                round(
                    pdf.stat().st_size /
                    (1024 * 1024),
                    3
                )
        })


dataset_inventory_df = None

try:

    import pandas as pd

    dataset_inventory_df = pd.DataFrame(
        dataset_inventory
    )

    dataset_inventory_df.to_csv(
        PACKAGE_DIR /
        "dataset_info" /
        "dataset_inventory.csv",
        index=False
    )

except Exception:

    pass


# --------------------------------------------------------------
# 8. CREATE DATASET README
# --------------------------------------------------------------

dataset_readme = PACKAGE_DIR / "dataset_info" / "README.txt"

with open(
    dataset_readme,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "NYAY AI — DATASET INFORMATION\n"
    )

    f.write(
        "=" * 60 + "\n\n"
    )

    f.write(
        f"Dataset path in Colab:\n"
        f"{DATASET_DIR}\n\n"
    )

    f.write(
        f"PDF files found: "
        f"{len(dataset_inventory)}\n\n"
    )

    f.write(
        "The official ChromaDB is included separately "
        "in the database/chroma_final folder.\n\n"
    )

    f.write(
        "The complete source PDFs are intentionally not "
        "duplicated inside this UI package.\n"
    )

# --------------------------------------------------------------
# 9. CREATE BACKEND STARTER
# --------------------------------------------------------------

backend_readme = PACKAGE_DIR / "backend" / "README.txt"

with open(
    backend_readme,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        """NYAY AI BACKEND
================

Expected backend architecture:

FastAPI
|
|-- /health
|-- /ask
|-- /upload
|-- /documents
|-- /citation
|
|-- Retriever Agent
|-- Answer Generation Agent
|-- Citation & Confidence Agent
|-- Orchestrator

The notebook contains the validated RAG pipeline.

For production deployment:
1. Load the official ChromaDB.
2. Keep user-upload ChromaDB separate.
3. Never hard-code API keys.
4. Use environment variables.
5. Return answer + citations + confidence to React.
"""
    )

# --------------------------------------------------------------
# 10. CREATE FRONTEND README
# --------------------------------------------------------------

frontend_readme = PACKAGE_DIR / "frontend" / "README.txt"

with open(
    frontend_readme,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        """NYAY AI FRONTEND
=================

Recommended UI:

1. Nyay AI logo / title
2. Legal question input
3. Ask button
4. Upload PDF button
5. Answer panel
6. Citation/source panel
7. Confidence indicator
8. Retrieved source information
9. Clear conversation button

Expected backend response:

{
    "answer": "...",
    "confidence": "Corpus-Grounded",
    "citations": [
        {
            "law_name": "...",
            "page": 4
        }
    ]
}

Important:
The frontend must NOT contain Gemini/Groq API keys.
"""
    )

# --------------------------------------------------------------
# 11. CREATE .ENV TEMPLATE
# --------------------------------------------------------------

env_template = PACKAGE_DIR / "config" / ".env.example"

with open(
    env_template,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        """# Nyay AI Environment Variables
# NEVER put real API keys in GitHub or ZIP files.

GEMINI_API_KEY=YOUR_GEMINI_API_KEY_HERE

# ChromaDB
CHROMA_PATH=./database/chroma_final

# User upload database
USER_CHROMA_PATH=./database/user_upload_chroma

# Collection
CHROMA_COLLECTION=nyay_ai_final

# API
BACKEND_URL=http://localhost:8000
"""
    )

# --------------------------------------------------------------
# 12. CREATE GITIGNORE
# --------------------------------------------------------------

gitignore = PACKAGE_DIR / "config" / ".gitignore"

with open(
    gitignore,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        """# Secrets
.env
.env.*
!.env.example

# Python
__pycache__/
*.pyc

# Node
node_modules/
dist/
build/

# Temporary files
*.tmp
*.log

# User data
user_uploads/

# Runtime
*.lock
"""
    )

# --------------------------------------------------------------
# 13. CREATE PROJECT README
# --------------------------------------------------------------

readme = PACKAGE_DIR / "README.md"

with open(
    readme,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        """# Nyay AI — Final UI / Project Package

## Project

Nyay AI is a Maharashtra-focused legal Retrieval-Augmented
Generation (RAG) assistant.

## Validated Evaluation

30 official questions were evaluated.

- Retrieval success: 30/30
- Precision@5: 76.67%
- Recall@5: 100.00%
- Mean similarity: 0.7364
- Mean retrieval latency: 0.7361 sec
- Citation accuracy: 83.33%
- Mean input tokens: 2259.87
- Mean output tokens: 184.73
- Mean total tokens: 2444.60
- Mean answer latency: 34.2237 sec

## Architecture

1. Ingestion Agent
2. Retriever Agent
3. Answer Generation Agent
4. Citation & Confidence Agent
5. Orchestrator

## Technology

- Python
- PyMuPDF
- LangChain
- ChromaDB
- MiniLM embeddings
- Gemini
- FastAPI
- React

## Important

API keys are NOT included.

Use:

config/.env.example

Create your own .env file and add the API key.

## Database

The validated ChromaDB is included in:

database/chroma_final

User-upload storage:

database/user_upload_chroma

## Evaluation

All final evaluation CSV files are included in:

evaluation/

## UI

The frontend should provide:

- Legal question input
- PDF upload
- Answer
- Citations
- Confidence
- Source information

## Do Not

- Do not expose API keys in React.
- Do not commit .env.
- Do not randomly rebuild the official ChromaDB.
- Do not replace the validated evaluation results.
"""
    )

# --------------------------------------------------------------
# 14. CREATE REQUIREMENTS
# --------------------------------------------------------------

requirements = PACKAGE_DIR / "config" / "requirements.txt"

with open(
    requirements,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        """pymupdf
langchain
langchain-community
langchain-huggingface
chromadb
sentence-transformers
google-genai
fastapi
uvicorn
python-multipart
pandas
numpy
"""
    )

# --------------------------------------------------------------
# 15. CREATE PACKAGE MANIFEST
# --------------------------------------------------------------

manifest = {
    "project": "Nyay AI",
    "created": datetime.now().isoformat(),
    "notebooks": copied_notebooks,
    "evaluation_files": copied_evaluation,
    "main_chromadb_included": main_db_copied,
    "user_upload_chromadb_included": user_db_copied,
    "dataset_pdf_count_found": len(dataset_inventory),
    "api_keys_included": False,
    "gemini_api_called_by_this_cell": False
}

with open(
    PACKAGE_DIR /
    "package_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )

# --------------------------------------------------------------
# 16. ZIP EVERYTHING
# --------------------------------------------------------------

if ZIP_FILE.exists():
    ZIP_FILE.unlink()

print("\nCreating ZIP...")

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in PACKAGE_DIR.rglob("*"):

        if file_path.is_file():

            arcname = file_path.relative_to(
                PACKAGE_DIR
            )

            zipf.write(
                file_path,
                arcname
            )


# --------------------------------------------------------------
# 17. ZIP SIZE
# --------------------------------------------------------------

zip_size_mb = (
    ZIP_FILE.stat().st_size /
    (1024 * 1024)
)

print("\n" + "=" * 80)
print("NYAY AI FINAL PACKAGE CREATED")
print("=" * 80)

print(
    f"\nZIP file:\n{ZIP_FILE}"
)

print(
    f"\nZIP size: {zip_size_mb:.2f} MB"
)

print(
    f"\nNotebooks: {len(copied_notebooks)}"
)

print(
    f"Evaluation files: {len(copied_evaluation)}"
)

print(
    f"Dataset PDFs inventoried: "
    f"{len(dataset_inventory)}"
)

print(
    f"Main ChromaDB included: "
    f"{main_db_copied}"
)

print(
    f"User-upload ChromaDB included: "
    f"{user_db_copied}"
)

print(
    "\nAPI keys included: NO"
)

print(
    "\nGemini API called by this cell: NO"
)

print("\n" + "=" * 80)
print("READY FOR UI / BACKEND DEVELOPMENT")
print("=" * 80)

print(
    f"\nDownload/copy ZIP from:"
)

print(
    ZIP_FILE
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PACKAGING NYAY AI

Notebooks copied: 0
Evaluation files copied: 8
Main ChromaDB copied: True
User-upload ChromaDB copied: True

Creating ZIP...

NYAY AI FINAL PACKAGE CREATED

ZIP file:
/content/drive/MyDrive/Nyay_AI/Nyay_AI_FINAL_UI_PACKAGE.zip

ZIP size: 43.63 MB

Notebooks: 0
Evaluation files: 8
Dataset PDFs inventoried: 65
Main ChromaDB included: True
User-upload ChromaDB included: True

API keys included: NO

Gemini API called by this cell: NO

READY FOR UI / BACKEND DEVELOPMENT

Download/copy ZIP from:
/content/drive/MyDrive/Nyay_AI/Nyay_AI_FINAL_UI_PACKAGE.zip


In [55]:
# Cell 31 — Identify Exact Judgment / GR Documents for Q26–Q30
# No Gemini/API call.

import os
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/datset 2")

pdf_files = sorted(DATASET_DIR.glob("*.pdf"))

print("=" * 70)
print("ALL CORPUS DOCUMENTS")
print("=" * 70)

print("Total PDFs:", len(pdf_files))
print()

for i, pdf in enumerate(pdf_files, 1):
    print(f"{i:02d}. {pdf.name}")

ALL CORPUS DOCUMENTS
Total PDFs: 0



In [ ]:
# Cell 29 — User-PDF human metrics
USER_REVIEW_PATH = EVALUATION_PATH / "user_20_gemini_answers.csv"

if USER_REVIEW_PATH.exists():
    reviewed = pd.read_csv(USER_REVIEW_PATH)
    if "human_answer_correct" in reviewed.columns:
        for col in ["human_answer_correct", "human_grounded", "human_citation_correct"]:
            reviewed[col] = reviewed[col].fillna("").astype(str).str.upper()

        for col in ["human_answer_correct", "human_grounded", "human_citation_correct"]:
            checked = reviewed[reviewed[col].isin(["YES", "NO"])]
            score = (
                (checked[col] == "YES").mean()
                if len(checked) else None
            )
            print(col, ":", None if score is None else round(score * 100, 2), "%")

        print("⚠️ These are only final metrics after you manually fill the YES/NO review columns.")


In [ ]:
# Cell 30 — Mixed-query demonstration report
assert "mixed_result" in globals(), "Run the mixed-query demo first."

mixed_report = pd.DataFrame([
    {
        "source_id": s["source_id"],
        "retrieval_source": s["retrieval_source"],
        "filename": s["filename"],
        "page": s["page"],
        "similarity": s["similarity"],
    }
    for s in mixed_result["citations"]
])

display(mixed_report)
print("Mixed answer:")
print(mixed_result["answer"])


,source_id,retrieval_source,filename,page,similarity
0,S1,official_corpus,The Maharashtra Agricultural Produce Marketing...,28,0.578433
1,S2,official_corpus,"Maharashtra Municipal Councils, Nagar Panchaya...",78,0.550276
2,S3,official_corpus,"Maharashtra Housing & Area Development Act, 19...",23,0.529461
3,S4,user_upload,20251107814584932.pdf,5,0.528645
4,S5,user_upload,20251107814584932 (1).pdf,5,0.528645


Mixed answer:
I could not find this information in the loaded documents.


In [ ]:
# Final cost calculation
# Cost is intentionally kept configurable because pricing can change.
# Set these values to the pricing applicable to the Gemini project/account
# before reporting cost in the paper.

INPUT_COST_PER_MILLION = None
OUTPUT_COST_PER_MILLION = None

print("Gemini provider:", LLM_MODEL)
print("Cost calculation is configurable; verify current Gemini pricing before paper reporting.")


Official evaluation cost: {'input_tokens': 5339, 'output_tokens': 815, 'cost_usd': 0.000644925}
⚠️ Verify the pricing shown in your Groq account before putting cost in the paper.


In [ ]:
# Cell 32 — Human-reviewed official answer/citation template
OFFICIAL_REVIEW_PATH = EVALUATION_PATH / "official_human_review.csv"

if not OFFICIAL_REVIEW_PATH.exists():
    review = official_answers_df.copy()
    review["human_answer_correct"] = ""
    review["human_grounded"] = ""
    review["human_citation_correct"] = ""
    review["review_notes"] = ""
    review.to_csv(OFFICIAL_REVIEW_PATH, index=False)
    print("Created:", OFFICIAL_REVIEW_PATH)
else:
    print("Review file already exists:", OFFICIAL_REVIEW_PATH)

print("Fill YES/NO only after checking each answer against the retrieved source text.")


Created: /content/drive/MyDrive/Nyay_AI/evaluation/official_human_review.csv
Fill YES/NO only after checking each answer against the retrieved source text.


In [ ]:
# Cell 33 — Compute final human-reviewed official metrics
if OFFICIAL_REVIEW_PATH.exists():
    rev = pd.read_csv(OFFICIAL_REVIEW_PATH)
    for col in ["human_answer_correct", "human_grounded", "human_citation_correct"]:
        if col not in rev.columns:
            rev[col] = ""
        rev[col] = rev[col].fillna("").astype(str).str.upper()

    for col in ["human_answer_correct", "human_grounded", "human_citation_correct"]:
        checked = rev[rev[col].isin(["YES", "NO"])]
        score = (checked[col] == "YES").mean() if len(checked) else None
        print(col, ":", "Not reviewed" if score is None else f"{score*100:.2f}%")


human_answer_correct : Not reviewed
human_grounded : Not reviewed
human_citation_correct : Not reviewed


In [ ]:
# Cell 34 — Final project summary
summary = {
    "project": "Nyay AI",
    "official_pdf_count": len(pdfs),
    "official_indexed_records": int(official_count),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": 384,
    "chunk_size_tokens": CHUNK_SIZE,
    "chunk_overlap_tokens": CHUNK_OVERLAP,
    "retrieval_method": "Cosine similarity",
    "top_k": TOP_K,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "llm_provider": "gemini",
    "llm_model": "gemini-3.6-flash",
    "official_evaluation_questions": len(official_eval),
    "user_evaluation_questions": 20,
    "precision_at_5_official": precision_at_5_official,
    "recall_at_5_official": recall_at_5_official,
    "average_official_retrieval_latency_seconds": float(official_retrieval_df["retrieval_latency_seconds"].mean()),
    "official_gemini_cost_usd": None,
    "user_upload_workspace": USER_WORKSPACE_ID,
    "mixed_query_supported": True,
}
SUMMARY_PATH = ARTIFACT_PATH / "final_project_summary.json"
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("Saved:", SUMMARY_PATH)


{
  "project": "Nyay AI",
  "official_pdf_count": 68,
  "official_indexed_records": 5259,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dimension": 384,
  "chunk_size_tokens": 500,
  "chunk_overlap_tokens": 50,
  "retrieval_method": "Cosine similarity",
  "top_k": 5,
  "confidence_threshold": 0.7,
  "llm_provider": "groq",
  "llm_model": "openai/gpt-oss-20b",
  "official_evaluation_questions": 30,
  "user_evaluation_questions": 20,
  "precision_at_5_official": null,
  "recall_at_5_official": null,
  "average_official_retrieval_latency_seconds": 0.3997666666666667,
  "official_groq_cost_usd": 0.000644925,
  "user_upload_workspace": "evaluation_user",
  "mixed_query_supported": true
}
Saved: /content/drive/MyDrive/Nyay_AI/artifacts/final_project_summary.json


In [ ]:
# Final health check — UI-ready Gemini configuration

checks = {
    "official ChromaDB non-empty": official_count > 0,
    "MiniLM model": EMBEDDING_MODEL == TOKENIZER_NAME,
    "384-d embeddings": len(probe) == 384,
    "chunk size = 500": CHUNK_SIZE == 500,
    "chunk overlap = 50": CHUNK_OVERLAP == 50,
    "top-k = 5": TOP_K == 5,
    "confidence threshold = 0.70": CONFIDENCE_THRESHOLD == 0.70,
    "Gemini provider": LLM_PROVIDER == "gemini",
    "Gemini 3.6 Flash": LLM_MODEL == "gemini-3.6-flash",
    "Official orchestrator": isinstance(official_orchestrator, NyayAIOrchestrator),
    "Unified flow available": "unified_orchestrator" in globals(),
}

for name, ok in checks.items():
    print(("OK" if ok else "FAIL"), name)

assert all(checks.values())
print("\nNYAY AI UI-READY NOTEBOOK HEALTH CHECK PASSED")


✅ 68 official PDFs
✅ official ChromaDB non-empty
✅ MiniLM model
✅ 384-d embeddings
✅ top-k = 5
✅ confidence threshold = 0.70
✅ Groq provider
✅ GPT-OSS 20B
✅ Official orchestrator
✅ Unified flow available

✅ NYAY AI FINAL NOTEBOOK HEALTH CHECK PASSED
